In [ ]:
!pip install biopython
!pip install xgboost
!pip install imbalanced-learn
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 3.8 MB/s eta 0:00:00


In [ ]:
from Bio import Entrez
import pandas as pd
import time
import torch

# Essential setup for NCBI
Entrez.email = "ktoolan@uri.edu"

In [ ]:
from Bio import Entrez, SeqIO

def fetch_protein_sequence(accession):
    Entrez.email = "ktoolan@uri.edu"  # 必须填写
    try:
        handle = Entrez.efetch(db="protein", id=accession, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        handle.close()
        return str(record.seq)
    except Exception as e:
        print(f"Error fetching {accession}: {e}")
        return None

In [ ]:
# =============================================================================
# Clean 400-virus mapping (original 51 + additional 149 + extra 200)
# =============================================================================

# 1. Original 51 entries (your first dictionary)
original_mapping = {
    "YP_232997": ["Orthopoxviruses",  "Vaccinia", "H3L", 0],
    "URK44321":  ["Orthopoxviruses",  "Mpox", "H3L", 1],
    "NP_042078": ["Orthopoxviruses",  "Smallpox", "H3L", 2],
    "NP_604441": ["Paramyxoviridae", "Human Parainfluenza 1", "F-Protein", 0],
    "NP_004680": ["Paramyxoviridae", "Measles (Edmonston)", "F-Protein", 1],
    "YP_009142751": ["Paramyxoviridae", "Mumps (Jeryl Lynn)", "F-Protein", 1],
    "NP_112026": ["Paramyxoviridae", "Nipah Virus", "F-Protein", 2],
    "NP_047111": ["Paramyxoviridae", "Hendra Virus", "F-Protein", 2],
    "NP_741961": ["Picornaviridae", "Coxsackievirus A9", "VP1", 0],
    "NP_740523": ["Picornaviridae", "Rhinovirus A", "VP1", 1],
    "NP_741975": ["Picornaviridae", "Enterovirus A71", "VP1", 1],
    "NP_041341": ["Picornaviridae", "Poliovirus", "VP1", 2],
    "NP_742055": ["Picornaviridae", "Enterovirus D68", "VP1", 2],
    "YP_003256193": ["Caliciviridae", "Sapovirus", "VP1", 0],
    "NP_056821": ["Caliciviridae", "Norwalk Virus", "VP1", 1],
    "NP_740333": ["Caliciviridae", "Rabbit Hemorrhagic Disease", "VP1", 2],
    "YP_009041935": ["Reoviridae", "Mammalian Orthoreovirus", "VP7", 0],
    "NP_694432": ["Reoviridae", "Rotavirus A", "VP7", 1],
    "NP_690853": ["Reoviridae", "Colorado Tick Fever", "VP7", 2],
    "NP_042931": ["Herpesviridae", "Human Herpesvirus 6", "gB-Protein", 0],
    "NP_044629": ["Herpesviridae", "Herpes Simplex 1 (HSV-1)", "gB-Protein", 1],
    "NP_040154": ["Herpesviridae", "Varicella-Zoster (Chickenpox)", "gB-Protein", 1],
    "YP_001956100": ["Herpesviridae", "B Virus (Macacine 1)", "gB-Protein", 2],
    "YP_081514": ["Herpesviridae", "Cytomegalovirus (CMV)", "gB-Protein", 2],
    "NP_073551": ["Coronaviruses", "Human Coronavirus 229E", "S-Protein", 0],
    "YP_009724390": ["Coronaviruses", "SARS-CoV-2 (Omicron)", "S-Protein", 1],
    "YP_009047204": ["Coronaviruses", "MERS-CoV", "S-Protein", 2],
    "NP_690583": ["Filoviruses", "Reston Virus", "GP-Protein", 0],
    "YP_003815426": ["Filoviruses", "Bundibugyo Ebola", "GP-Protein", 1],
    "NP_051149": ["Filoviruses", "Zaire Ebola", "GP-Protein", 2],
    "NP_040304": ["Papillomaviridae", "HPV Type 1", "L1-Protein", 0],
    "AZI94870.1": ["Papillomaviridae", "HPV Type 6", "L1-Protein", 1],
    "NP_041332": ["Papillomaviridae", "HPV Type 16", "L1-Protein", 2],
    "YP_068027": ["Adenoviridae", "Human Mastadenovirus A", "Hexon", 0],
    "NP_040523": ["Adenoviridae", "Human Mastadenovirus C", "Hexon", 1],
    "YP_068042": ["Adenoviridae", "Human Mastadenovirus B", "Hexon", 2],
    "YP_009333256": ["Rhabdoviridae", "Perch Rhabdovirus", "G-Protein", 0],
    "XFF06267": ["Rhabdoviridae", "Chandipura Virus", "G-Protein", 1],
    "NP_056796": ["Rhabdoviridae", "Rabies Virus", "G-Protein", 2],
    "NP_941983": ["Hantaviridae", "Prospect Hill Virus", "Gp-Protein", 0],
    "NP_941989": ["Hantaviridae", "Puumala Hantavirus", "Gp-Protein", 1],
    "NP_941974": ["Hantaviridae", "Sin Nombre Hantavirus", "Gp-Protein", 2],
    "NP_062884": ["Togaviridae", "Rubella Virus (Vaccine)", "E1-Protein", 0],
    "YP_001427553": ["Togaviridae", "Chikungunya Virus", "E1-Protein", 1],
    "NP_740645": ["Togaviridae", "Eastern Equine Encephalitis", "E1-Protein", 2],
    "NP_694870": ["Arenaviridae", "Tacaribe Virus", "GPC-Protein", 0],
    "NP_694851": ["Arenaviridae", "Lymphocytic Choriomeningitis", "GPC-Protein", 1],
    "NP_694872": ["Arenaviridae", "Lassa Virus", "GPC-Protein", 2],
    "NP_044927": ["Parvoviridae", "Adeno-associated 2 (AAV)", "VP2", 0],
    "NP_040873": ["Parvoviridae", "Parvovirus B19", "VP2", 1],
    "NP_042340": ["Parvoviridae", "Canine Parvovirus", "VP2", 2],
}

# 2. Additional 149 (first expansion batch)
additional_149 = {
    "NP_619651": ["Orthopoxviruses", "Cowpox Virus", "H3L", 1],
    "YP_009143672": ["Orthopoxviruses", "Camelpox Virus", "H3L", 1],
    "NP_619647": ["Orthopoxviruses", "Ectromelia Virus", "H3L", 1],
    "YP_009143655": ["Orthopoxviruses", "Taterapox Virus", "H3L", 1],
    "YP_009108769": ["Orthopoxviruses", "Raccoonpox Virus", "H3L", 0],
    "YP_009109051": ["Orthopoxviruses", "Skunkpox Virus", "H3L", 0],
    "NP_042160":  ["Orthopoxviruses", "Rabbitpox Virus", "H3L", 1],
    "YP_009112849": ["Orthopoxviruses", "Volepox Virus", "H3L", 0],
    "YP_009109149": ["Orthopoxviruses", "Gerbilpox Virus", "H3L", 0],
    "YP_009112909": ["Orthopoxviruses", "Squirrelpox Virus", "H3L", 0],
    "NP_112025":  ["Paramyxoviridae", "Hendra Virus (Hendra)", "F-Protein", 2],
    "NP_112024":  ["Paramyxoviridae", "Nipah Virus (Malaysia)", "F-Protein", 2],
    "NP_112027":  ["Paramyxoviridae", "Nipah Virus (Bangladesh)", "F-Protein", 2],
    "YP_009142752": ["Paramyxoviridae", "Mumps (RIT 4385)", "F-Protein", 1],
    "YP_009142753": ["Paramyxoviridae", "Mumps (L-Zagreb)", "F-Protein", 1],
    "NP_604442":  ["Paramyxoviridae", "Human Parainfluenza 2", "F-Protein", 0],
    "NP_604443":  ["Paramyxoviridae", "Human Parainfluenza 3", "F-Protein", 1],
    "NP_598240":  ["Paramyxoviridae", "Human Parainfluenza 4a", "F-Protein", 0],
    "NP_598241":  ["Paramyxoviridae", "Simian Virus 5", "F-Protein", 0],
    "NP_598242":  ["Paramyxoviridae", "Canine Distemper Virus", "F-Protein", 1],
    "NP_047112":  ["Paramyxoviridae", "Newcastle Disease Virus", "F-Protein", 0],
    "YP_009357029": ["Paramyxoviridae", "Sosuga Virus", "F-Protein", 2],
    "NP_740524":  ["Picornaviridae", "Rhinovirus B", "VP1", 1],
    "NP_740525":  ["Picornaviridae", "Rhinovirus C", "VP1", 1],
    "NP_740526":  ["Picornaviridae", "Coxsackievirus B1", "VP1", 1],
    "NP_740527":  ["Picornaviridae", "Coxsackievirus B3", "VP1", 1],
    "NP_740528":  ["Picornaviridae", "Coxsackievirus B5", "VP1", 1],
    "NP_740529":  ["Picornaviridae", "Echovirus 6", "VP1", 1],
    "NP_740530":  ["Picornaviridae", "Echovirus 9", "VP1", 1],
    "NP_740531":  ["Picornaviridae", "Echovirus 11", "VP1", 1],
    "NP_740532":  ["Picornaviridae", "Enterovirus C (CVA21)", "VP1", 1],
    "NP_740533":  ["Picornaviridae", "Enterovirus D94", "VP1", 1],
    "NP_740534":  ["Picornaviridae", "Enterovirus A89", "VP1", 1],
    "NP_740331":  ["Caliciviridae", "Norovirus GII.4", "VP1", 1],
    "NP_740332":  ["Caliciviridae", "Norovirus GI.1", "VP1", 1],
    "NP_056820":  ["Caliciviridae", "Sapporo Virus", "VP1", 0],
    "YP_003256194":["Caliciviridae", "Feline Calicivirus", "VP1", 0],
    "YP_009342921":["Caliciviridae", "Vesicular Exanthema Virus", "VP1", 0],
    "YP_009238961":  ["Caliciviridae", "Murine Norovirus", "VP1", 0],
    "NP_694433":  ["Reoviridae", "Rotavirus B", "VP7", 1],
    "NP_694434":  ["Reoviridae", "Rotavirus C", "VP7", 1],
    "NP_690854":  ["Reoviridae", "Great Island Virus", "VP7", 0],
    "YP_009041936":["Reoviridae", "Avian Orthoreovirus", "VP7", 0],
    "YP_009041937":["Reoviridae", "Baboon Orthoreovirus", "VP7", 0],
    "NP_690855":  ["Reoviridae", "Kemerovo Virus", "VP7", 2],
    "NP_044630":  ["Herpesviridae", "Herpes Simplex 2 (HSV-2)", "gB-Protein", 1],
    "NP_044631":  ["Herpesviridae", "Epstein-Barr Virus", "gB-Protein", 1],
    "NP_044632":  ["Herpesviridae", "Human Cytomegalovirus (Toledo)", "gB-Protein", 2],
    "YP_081515":  ["Herpesviridae", "Cytomegalovirus (Guinea pig)", "gB-Protein", 0],
    "YP_001956101":["Herpesviridae", "Cercopithecine Herpesvirus 1", "gB-Protein", 2],
    "YP_001956102":["Herpesviridae", "Human Herpesvirus 8", "gB-Protein", 1],
    "NP_042932":  ["Herpesviridae", "Human Herpesvirus 7", "gB-Protein", 0],
    "YP_009042457":["Herpesviridae", "Saimiriine Herpesvirus 2", "gB-Protein", 0],
    "NP_044633":  ["Herpesviridae", "Equine Herpesvirus 1", "gB-Protein", 0],
    "YP_009042458":["Herpesviridae", "Pseudorabies Virus", "gB-Protein", 0],
    "YP_009724391":["Coronaviruses", "SARS-CoV-2 (Delta)", "S-Protein", 1],
    "YP_009724392":["Coronaviruses", "SARS-CoV-2 (Alpha)", "S-Protein", 1],
    "YP_009724393":["Coronaviruses", "SARS-CoV-2 (Beta)", "S-Protein", 1],
    "YP_009047205":["Coronaviruses", "SARS-CoV-1", "S-Protein", 2],
    "YP_009555255":["Coronaviruses", "MERS-CoV (England1)", "S-Protein", 2],
    "NP_073550":  ["Coronaviruses", "Human Coronavirus OC43", "S-Protein", 0],
    "NP_073549":  ["Coronaviruses", "Human Coronavirus NL63", "S-Protein", 0],
    "YP_009333561":["Coronaviruses", "Human Coronavirus HKU1", "S-Protein", 0],
    "YP_009072445":["Coronaviruses", "Bat Coronavirus HKU5", "S-Protein", 2],
    "YP_009072446":["Coronaviruses", "Bat Coronavirus HKU9", "S-Protein", 0],
    "NP_051150":  ["Filoviruses", "Sudan Ebola Virus", "GP-Protein", 2],
    "NP_051151":  ["Filoviruses", "Tai Forest Ebola Virus", "GP-Protein", 2],
    "YP_003815427":["Filoviruses", "Marburg Virus (Musoke)", "GP-Protein", 2],
    "YP_003815428":["Filoviruses", "Marburg Virus (Ravn)", "GP-Protein", 2],
    "YP_009343161":["Filoviruses", "Lloviu Virus", "GP-Protein", 2],
    "NP_690584":  ["Filoviruses", "Reston Virus (Pennsylvania)", "GP-Protein", 0],
    "NP_041333":  ["Papillomaviridae", "HPV Type 18", "L1-Protein", 2],
    "NP_041334":  ["Papillomaviridae", "HPV Type 31", "L1-Protein", 2],
    "NP_041335":  ["Papillomaviridae", "HPV Type 33", "L1-Protein", 2],
    "NP_041336":  ["Papillomaviridae", "HPV Type 45", "L1-Protein", 2],
    "NP_040305":  ["Papillomaviridae", "HPV Type 2", "L1-Protein", 1],
    "NP_040306":  ["Papillomaviridae", "HPV Type 3", "L1-Protein", 1],
    "NP_040307":  ["Papillomaviridae", "HPV Type 4", "L1-Protein", 1],
    "NP_040308":  ["Papillomaviridae", "HPV Type 7", "L1-Protein", 1],
    "NP_041337":  ["Papillomaviridae", "HPV Type 52", "L1-Protein", 2],
    "NP_041338":  ["Papillomaviridae", "HPV Type 58", "L1-Protein", 2],
    "YP_068028":  ["Adenoviridae", "Human Mastadenovirus D", "Hexon", 0],
    "YP_068029":  ["Adenoviridae", "Human Mastadenovirus E", "Hexon", 0],
    "YP_068030":  ["Adenoviridae", "Human Mastadenovirus F", "Hexon", 0],
    "YP_068031":  ["Adenoviridae", "Human Mastadenovirus G", "Hexon", 0],
    "YP_009480713":["Adenoviridae", "Simian Adenovirus 1", "Hexon", 0],
    "YP_009480714":["Adenoviridae", "Canine Adenovirus 1", "Hexon", 0],
    "YP_009480715":["Adenoviridae", "Fowl Adenovirus A", "Hexon", 0],
    "YP_009480716":["Adenoviridae", "Bovine Adenovirus B", "Hexon", 0],
    "YP_009480717":["Adenoviridae", "Porcine Adenovirus A", "Hexon", 0],
    "YP_009480718":["Adenoviridae", "Ovine Adenovirus D", "Hexon", 0],
    "NP_056797":  ["Rhabdoviridae", "Rabies Virus (CVS)", "G-Protein", 2],
    "NP_056798":  ["Rhabdoviridae", "Rabies Virus (ERA)", "G-Protein", 2],
    "NP_056799":  ["Rhabdoviridae", "Mokola Virus", "G-Protein", 2],
    "NP_056800":  ["Rhabdoviridae", "Lagos Bat Virus", "G-Protein", 2],
    "NP_056801":  ["Rhabdoviridae", "Duvenhage Virus", "G-Protein", 2],
    "YP_009333257":["Rhabdoviridae", "Vesicular Stomatitis Indiana Virus", "G-Protein", 0],
    "YP_009333258":["Rhabdoviridae", "Vesicular Stomatitis New Jersey Virus", "G-Protein", 0],
    "XFF06268":   ["Rhabdoviridae", "Chandipura Virus (Nagpur)", "G-Protein", 1],
    "YP_009333259":["Rhabdoviridae", "Isfahan Virus", "G-Protein", 1],
    "NP_941975":  ["Hantaviridae", "Andes Virus", "Gp-Protein", 2],
    "NP_941976":  ["Hantaviridae", "Hantaan Virus", "Gp-Protein", 1],
    "NP_941977":  ["Hantaviridae", "Seoul Virus", "Gp-Protein", 1],
    "NP_941978":  ["Hantaviridae", "Dobrava-Belgrade Virus", "Gp-Protein", 2],
    "NP_941979":  ["Hantaviridae", "Saaremaa Virus", "Gp-Protein", 1],
    "NP_941980":  ["Hantaviridae", "Thailand Virus", "Gp-Protein", 1],
    "NP_941981":  ["Hantaviridae", "Tula Virus", "Gp-Protein", 0],
    "NP_941982":  ["Hantaviridae", "Prospect Hill Virus (PHV)", "Gp-Protein", 0],
    "NP_941984":  ["Hantaviridae", "New York Virus", "Gp-Protein", 2],
    "NP_740646":  ["Togaviridae", "Venezuelan Equine Encephalitis Virus", "E1-Protein", 2],
    "NP_740647":  ["Togaviridae", "Western Equine Encephalitis Virus", "E1-Protein", 2],
    "NP_740648":  ["Togaviridae", "Sindbis Virus", "E1-Protein", 1],
    "NP_740649":  ["Togaviridae", "Semliki Forest Virus", "E1-Protein", 1],
    "NP_062885":  ["Togaviridae", "Rubella Virus (M33)", "E1-Protein", 0],
    "YP_001427554":["Togaviridae", "O'nyong-nyong Virus", "E1-Protein", 1],
    "YP_001427555":["Togaviridae", "Ross River Virus", "E1-Protein", 1],
    "NP_740650":  ["Togaviridae", "Barmah Forest Virus", "E1-Protein", 1],
    "NP_694873":  ["Arenaviridae", "Lassa Virus (Josiah)", "GPC-Protein", 2],
    "NP_694852":  ["Arenaviridae", "Lymphocytic Choriomeningitis Virus (Armstrong)", "GPC-Protein", 1],
    "NP_694871":  ["Arenaviridae", "Junin Virus", "GPC-Protein", 2],
    "NP_694874":  ["Arenaviridae", "Machupo Virus", "GPC-Protein", 2],
    "NP_694875":  ["Arenaviridae", "Guanarito Virus", "GPC-Protein", 2],
    "NP_694876":  ["Arenaviridae", "Sabia Virus", "GPC-Protein", 2],
    "NP_694877":  ["Arenaviridae", "Pichinde Virus", "GPC-Protein", 0],
    "NP_694878":  ["Arenaviridae", "Tamiami Virus", "GPC-Protein", 0],
    "NP_040874":  ["Parvoviridae", "Human Bocavirus", "VP2", 1],
    "NP_040875":  ["Parvoviridae", "Porcine Parvovirus", "VP2", 0],
    "NP_040876":  ["Parvoviridae", "Feline Panleukopenia Virus", "VP2", 0],
    "NP_040877":  ["Parvoviridae", "Mink Enteritis Virus", "VP2", 0],
    "NP_040878":  ["Parvoviridae", "Canine Parvovirus 2a", "VP2", 2],
    "NP_040879":  ["Parvoviridae", "Canine Parvovirus 2b", "VP2", 2],
    "NP_044928":  ["Parvoviridae", "Adeno-associated 3", "VP2", 0],
    "NP_044929":  ["Parvoviridae", "Adeno-associated 5", "VP2", 0],
    "NP_044930":  ["Parvoviridae", "Adeno-associated 8", "VP2", 0],
    # Orthopoxviruses (H3L)
    "YP_009143675": ["Orthopoxviruses", "Camelpox Virus (CP-19)", "H3L", 1],
    "YP_009143676": ["Orthopoxviruses", "Cowpox Virus (Brighton)", "H3L", 1],
    "YP_009143677": ["Orthopoxviruses", "Ectromelia Virus (MP-3)", "H3L", 1],
    "YP_009143678": ["Orthopoxviruses", "Monkeypox Virus (Zaire-96)", "H3L", 1],

    # Paramyxoviridae (F protein)
    "NP_598249":  ["Paramyxoviridae", "Sendai Virus", "F-Protein", 0],
    "NP_598250":  ["Paramyxoviridae", "Human Parainfluenza 4b", "F-Protein", 0],
    "YP_009357038": ["Paramyxoviridae", "Nipah Virus (India)", "F-Protein", 2],
    "YP_009357039": ["Paramyxoviridae", "Hendra Virus (Redlands)", "F-Protein", 2],
    "NP_047116":  ["Paramyxoviridae", "Avian Paramyxovirus 6", "F-Protein", 0],
    "NP_598251":  ["Paramyxoviridae", "Porcine Rubulavirus", "F-Protein", 0],

    # Picornaviridae (VP1)
    "NP_740561":  ["Picornaviridae", "Coxsackievirus A12", "VP1", 1],
    "NP_740562":  ["Picornaviridae", "Coxsackievirus A14", "VP1", 1],
    "NP_740563":  ["Picornaviridae", "Echovirus 18", "VP1", 1],
    "NP_740564":  ["Picornaviridae", "Enterovirus B (Coxsackie B6)", "VP1", 1],
    "NP_740565":  ["Picornaviridae", "Human Rhinovirus 39", "VP1", 1],
    "NP_740566":  ["Picornaviridae", "Parechovirus B", "VP1", 1],

    # Caliciviridae (VP1)
    "NP_786893":  ["Caliciviridae", "Norovirus GII.1", "VP1", 1],
    "NP_786894":  ["Caliciviridae", "Norovirus GII.5", "VP1", 1],
    "YP_009342930": ["Caliciviridae", "Norovirus GII.11", "VP1", 1],
    "YP_009342931": ["Caliciviridae", "Sapporo-like virus (Tokyo)", "VP1", 0],
    "NP_786895":  ["Caliciviridae", "European Brown Hare Syndrome Virus", "VP1", 0],

    # Reoviridae (VP7)
    "NP_694445":  ["Reoviridae", "Rotavirus G5P[8]", "VP7", 1],
    "NP_694446":  ["Reoviridae", "Rotavirus G10P[11]", "VP7", 1],
    "NP_690859":  ["Reoviridae", "Kadipiro Virus", "VP7", 2],
    "YP_009041942": ["Reoviridae", "Mammalian Orthoreovirus (T1L)", "VP7", 0],
    "YP_009041943": ["Reoviridae", "Avian Orthoreovirus (176)", "VP7", 0],

    # Herpesviridae (gB)
    "YP_009042475": ["Herpesviridae", "Human Herpesvirus 1 (KOS)", "gB-Protein", 1],
    "YP_009042476": ["Herpesviridae", "Human Herpesvirus 2 (333)", "gB-Protein", 1],
    "YP_009042477": ["Herpesviridae", "Equid Herpesvirus 3", "gB-Protein", 0],
    "YP_081519":  ["Herpesviridae", "Bovine Herpesvirus 4", "gB-Protein", 0],
    "YP_009042478": ["Herpesviridae", "Gallid Herpesvirus 3", "gB-Protein", 0],

    # Coronaviruses (S)
    "YP_009724401": ["Coronaviruses", "SARS-CoV-2 (Kappa)", "S-Protein", 1],
    "YP_009724402": ["Coronaviruses", "SARS-CoV-2 (Eta)", "S-Protein", 1],
    "YP_009072452": ["Coronaviruses", "Bat Coronavirus HKU5-2", "S-Protein", 0],
    "YP_009333567": ["Coronaviruses", "Avian Coronavirus", "S-Protein", 0],
    "YP_009555262": ["Coronaviruses", "MERS-CoV (Bisha)", "S-Protein", 2],
    "YP_009333568": ["Coronaviruses", "Beluga Whale Coronavirus", "S-Protein", 0],

    # Filoviruses (GP)
    "YP_003815433": ["Filoviruses", "Marburg Virus (Ozolin)", "GP-Protein", 2],
    "YP_009343172": ["Filoviruses", "Ebola Virus (Bouee)", "GP-Protein", 2],
    "YP_009343173": ["Filoviruses", "Sudan Ebola Virus (Maleo)", "GP-Protein", 2],
    "NP_690588":  ["Filoviruses", "Reston Virus (Texas)", "GP-Protein", 0],
    "YP_009343174": ["Filoviruses", "Lloviu Virus (Hungary)", "GP-Protein", 0],

    # Papillomaviridae (L1)
    "NP_041356":  ["Papillomaviridae", "HPV Type 61", "L1-Protein", 2],
    "NP_041357":  ["Papillomaviridae", "HPV Type 72", "L1-Protein", 2],
    "NP_040319":  ["Papillomaviridae", "HPV Type 54", "L1-Protein", 1],
    "NP_040320":  ["Papillomaviridae", "HPV Type 62", "L1-Protein", 1],
    "NP_040321":  ["Papillomaviridae", "HPV Type 71", "L1-Protein", 1],

    # Adenoviridae (Hexon)
    "YP_009480740": ["Adenoviridae", "Human Mastadenovirus 26", "Hexon", 0],
    "YP_009480741": ["Adenoviridae", "Human Mastadenovirus 35", "Hexon", 0],
    "YP_009480742": ["Adenoviridae", "Simian Adenovirus 22", "Hexon", 0],
    "YP_009480743": ["Adenoviridae", "Canine Adenovirus 2 (Toronto)", "Hexon", 0],
    "YP_009480744": ["Adenoviridae", "Bovine Adenovirus D", "Hexon", 0],

    # Rhabdoviridae (G)
    "NP_056815":  ["Rhabdoviridae", "Gannoruwa Bat Lyssavirus", "G-Protein", 2],
    "YP_009333269": ["Rhabdoviridae", "Vesicular Stomatitis Indiana Virus (Glasgow)", "G-Protein", 0],
    "YP_009333270": ["Rhabdoviridae", "Chandipura Virus (Ghana)", "G-Protein", 1],
    "NP_056816":  ["Rhabdoviridae", "Taiwan Bat Lyssavirus", "G-Protein", 2],

    # Hantaviridae (Gp)
    "NP_942012":  ["Hantaviridae", "Maporal Virus", "Gp-Protein", 1],
    "NP_942013":  ["Hantaviridae", "Montano Virus", "Gp-Protein", 1],
    "NP_942014":  ["Hantaviridae", "Necocli Virus", "Gp-Protein", 1],
    "NP_942015":  ["Hantaviridae", "Prospect Hill (PH-1)", "Gp-Protein", 0],
    "NP_942016":  ["Hantaviridae", "Caño Delgadito (CDG-1)", "Gp-Protein", 1],

    # Togaviridae (E1)
    "NP_740665":  ["Togaviridae", "Ijanga Virus", "E1-Protein", 1],
    "NP_740666":  ["Togaviridae", "Kyzylagach Virus", "E1-Protein", 1],
    "YP_001427559": ["Togaviridae", "Middelburg Virus", "E1-Protein", 1],
    "NP_740667":  ["Togaviridae", "Ndumu Virus", "E1-Protein", 1],
    "NP_740668":  ["Togaviridae", "Triniti Virus", "E1-Protein", 1],

    # Arenaviridae (GPC)
    "NP_694900":  ["Arenaviridae", "Dandenong Virus", "GPC-Protein", 1],
    "NP_694901":  ["Arenaviridae", "Gairo Virus", "GPC-Protein", 1],
    "NP_694907":  ["Arenaviridae", "Kodoko Virus", "GPC-Protein", 1],
    "NP_694903":  ["Arenaviridae", "Loei River Virus", "GPC-Protein", 1],
    "NP_694904":  ["Arenaviridae", "Menekre Virus", "GPC-Protein", 1],

    # Parvoviridae (VP2)
    "NP_040893":  ["Parvoviridae", "Canine Parvovirus 2c (Asia)", "VP2", 2],
    "NP_044940":  ["Parvoviridae", "Adeno-associated 13", "VP2", 0],
    "NP_040894":  ["Parvoviridae", "Bovine Parvovirus 2", "VP2", 0],
    "NP_044941":  ["Parvoviridae", "Snake Parvovirus", "VP2", 0],
    "NP_040895":  ["Parvoviridae", "Porcine Parvovirus 2", "VP2", 0],
}

# 3. Extra 200 (second expansion batch)
extra_200 = {
    "NP_042041":  ["Orthopoxviruses", "Variola Virus (India-1967)", "H3L", 2],
    "YP_009143671":["Orthopoxviruses", "Camelpox Virus (CP-1)", "H3L", 1],
    "NP_619650":  ["Orthopoxviruses", "Cowpox Virus (Brighton Red)", "H3L", 1],
    "YP_009143673":["Orthopoxviruses", "Monkeypox Virus (Liberia)", "H3L", 1],
    "YP_009109063":["Orthopoxviruses", "Vaccinia Virus (Copenhagen)", "H3L", 0],
    "YP_009108799":["Orthopoxviruses", "Raccoonpox Virus (Herman)", "H3L", 0],
    "NP_042148":  ["Orthopoxviruses", "Variola Virus (Bangladesh)", "H3L", 2],
    "NP_619652":  ["Orthopoxviruses", "Ectromelia Virus (Moscow)", "H3L", 1],
    "YP_009112881":["Orthopoxviruses", "Taterapox Virus (Dahomey)", "H3L", 1],
    "YP_009112923":["Orthopoxviruses", "Volepox Virus (California)", "H3L", 0],
    "NP_112028":  ["Paramyxoviridae", "Tioman Virus", "F-Protein", 1],
    "NP_112029":  ["Paramyxoviridae", "Menangle Virus", "F-Protein", 1],
    "NP_598243":  ["Paramyxoviridae", "Mapuera Virus", "F-Protein", 1],
    "YP_009357030":["Paramyxoviridae", "Mojiang Virus", "F-Protein", 2],
    "NP_047113":  ["Paramyxoviridae", "Avian Paramyxovirus 1", "F-Protein", 0],
    "NP_598244":  ["Paramyxoviridae", "Bovine Parainfluenza 3", "F-Protein", 0],
    "YP_009142750":["Paramyxoviridae", "Mumps Virus (SBL)", "F-Protein", 1],
    "YP_009508961":  ["Paramyxoviridae", "Simian Virus 41", "F-Protein", 0],
    "YP_009357031":["Paramyxoviridae", "Cedar Virus", "F-Protein", 2],
    "YP_009508959":  ["Paramyxoviridae", "Peste-des-petits-ruminants Virus", "F-Protein", 1],
    "NP_740535":  ["Picornaviridae", "Coxsackievirus A16", "VP1", 1],
    "NP_740536":  ["Picornaviridae", "Coxsackievirus A24", "VP1", 1],
    "NP_740537":  ["Picornaviridae", "Echovirus 30", "VP1", 1],
    "NP_740538":  ["Picornaviridae", "Enterovirus B (Echo 7)", "VP1", 1],
    "NP_740539":  ["Picornaviridae", "Enterovirus D70", "VP1", 1],
    "NP_740540":  ["Picornaviridae", "Human Rhinovirus 14", "VP1", 1],
    "NP_740541":  ["Picornaviridae", "Human Rhinovirus 16", "VP1", 1],
    "NP_740542":  ["Picornaviridae", "Parechovirus A", "VP1", 1],
    "NP_740543":  ["Picornaviridae", "Aichi Virus", "VP1", 0],
    "NP_740544":  ["Picornaviridae", "Bovine Enterovirus", "VP1", 0],
    "NP_786883":  ["Caliciviridae", "Norovirus GII.2", "VP1", 1],
    "NP_786884":  ["Caliciviridae", "Norovirus GII.17", "VP1", 1],
    "YP_009342922":["Caliciviridae", "San Miguel Sea Lion Virus", "VP1", 0],
    "NP_786885":  ["Caliciviridae", "Porcine Norovirus", "VP1", 0],
    "YP_009342923":["Caliciviridae", "Rabbit Calicivirus (RCV)", "VP1", 0],
    "NP_786886":  ["Caliciviridae", "Tulane Virus", "VP1", 0],
    "NP_694435":  ["Reoviridae", "Rotavirus G1P[8]", "VP7", 1],
    "NP_694436":  ["Reoviridae", "Rotavirus G2P[4]", "VP7", 1],
    "NP_694437":  ["Reoviridae", "Rotavirus G3P[8]", "VP7", 1],
    "NP_694438":  ["Reoviridae", "Rotavirus G4P[8]", "VP7", 1],
    "NP_690856":  ["Reoviridae", "Banna Virus", "VP7", 2],
    "YP_009041938":["Reoviridae", "Mammalian Orthoreovirus (Lang)", "VP7", 0],
    "YP_009042459":["Herpesviridae", "Bovine Herpesvirus 1", "gB-Protein", 0],
    "YP_009042460":["Herpesviridae", "Equine Herpesvirus 4", "gB-Protein", 0],
    "YP_009042461":["Herpesviridae", "Gallid Herpesvirus 2 (Marek)", "gB-Protein", 0],
    "YP_009042462":["Herpesviridae", "Infectious Laryngotracheitis Virus", "gB-Protein", 0],
    "NP_044634":  ["Herpesviridae", "Human Herpesvirus 4 (EBV B95-8)", "gB-Protein", 1],
    "NP_044635":  ["Herpesviridae", "Human Herpesvirus 5 (AD169)", "gB-Protein", 2],
    "YP_081516":  ["Herpesviridae", "Murine Cytomegalovirus", "gB-Protein", 0],
    "YP_009042463":["Herpesviridae", "Saimiriine Herpesvirus 1", "gB-Protein", 0],
    "YP_009724394":["Coronaviruses", "SARS-CoV-2 (Gamma)", "S-Protein", 1],
    "YP_009724395":["Coronaviruses", "SARS-CoV-2 (Epsilon)", "S-Protein", 1],
    "YP_009072447":["Coronaviruses", "Bat Coronavirus HKU4", "S-Protein", 2],
    "NP_073552":  ["Coronaviruses", "Porcine Epidemic Diarrhea Virus", "S-Protein", 0],
    "YP_009555256":["Coronaviruses", "MERS-CoV (Jordan-N3)", "S-Protein", 2],
    "NP_073553":  ["Coronaviruses", "Canine Coronavirus", "S-Protein", 0],
    "YP_009333562":["Coronaviruses", "Bovine Coronavirus", "S-Protein", 0],
    "YP_009072448":["Coronaviruses", "Bulbul Coronavirus HKU11", "S-Protein", 0],
    "YP_009555257":["Coronaviruses", "MERS-CoV (Al-Hasa)", "S-Protein", 2],
    "NP_073554":  ["Coronaviruses", "Feline Coronavirus", "S-Protein", 0],
    "YP_003815429":["Filoviruses", "Marburg Virus (Angola)", "GP-Protein", 2],
    "YP_003815430":["Filoviruses", "Marburg Virus (Ci67)", "GP-Protein", 2],
    "YP_009343162":["Filoviruses", "Bombali Virus", "GP-Protein", 2],
    "NP_690585":  ["Filoviruses", "Reston Virus (Siena)", "GP-Protein", 0],
    "YP_009343163":["Filoviruses", "Mengla Virus", "GP-Protein", 2],
    "NP_051152":  ["Filoviruses", "Ebola Virus (Mayinga)", "GP-Protein", 2],
    "NP_041339":  ["Papillomaviridae", "HPV Type 35", "L1-Protein", 2],
    "NP_041340":  ["Papillomaviridae", "HPV Type 39", "L1-Protein", 2],
    "NP_041342":  ["Papillomaviridae", "HPV Type 56", "L1-Protein", 2],
    "NP_041343":  ["Papillomaviridae", "HPV Type 59", "L1-Protein", 2],
    "NP_041344":  ["Papillomaviridae", "HPV Type 66", "L1-Protein", 2],
    "NP_041345":  ["Papillomaviridae", "HPV Type 68", "L1-Protein", 2],
    "NP_041346":  ["Papillomaviridae", "HPV Type 70", "L1-Protein", 1],
    "NP_040309":  ["Papillomaviridae", "HPV Type 10", "L1-Protein", 1],
    "NP_040310":  ["Papillomaviridae", "HPV Type 11", "L1-Protein", 1],
    "NP_040311":  ["Papillomaviridae", "HPV Type 13", "L1-Protein", 1],
    "YP_009480719":["Adenoviridae", "Human Mastadenovirus 3", "Hexon", 1],
    "YP_009480720":["Adenoviridae", "Human Mastadenovirus 5", "Hexon", 1],
    "YP_009480721":["Adenoviridae", "Human Mastadenovirus 7", "Hexon", 1],
    "YP_009480722":["Adenoviridae", "Human Mastadenovirus 14", "Hexon", 1],
    "YP_009480723":["Adenoviridae", "Simian Adenovirus 21", "Hexon", 0],
    "YP_009480724":["Adenoviridae", "Canine Adenovirus 2", "Hexon", 0],
    "YP_009480725":["Adenoviridae", "Bovine Adenovirus A", "Hexon", 0],
    "YP_068043":  ["Adenoviridae", "Human Mastadenovirus 21", "Hexon", 0],
    "NP_056802":  ["Rhabdoviridae", "European Bat Lyssavirus 1", "G-Protein", 2],
    "NP_056803":  ["Rhabdoviridae", "European Bat Lyssavirus 2", "G-Protein", 2],
    "NP_056804":  ["Rhabdoviridae", "Australian Bat Lyssavirus", "G-Protein", 2],
    "NP_056805":  ["Rhabdoviridae", "Irkut Virus", "G-Protein", 2],
    "YP_009333260":["Rhabdoviridae", "Chandipura Virus (CIN0451)", "G-Protein", 1],
    "YP_009333261":["Rhabdoviridae", "Vesicular Stomatitis Alagoas Virus", "G-Protein", 0],
    "NP_056806":  ["Rhabdoviridae", "Bokeloh Bat Lyssavirus", "G-Protein", 2],
    "YP_009333262":["Rhabdoviridae", "Spring Viremia of Carp Virus", "G-Protein", 0],
    "NP_941985":  ["Hantaviridae", "Bayou Virus", "Gp-Protein", 2],
    "NP_941986":  ["Hantaviridae", "Black Creek Canal Virus", "Gp-Protein", 2],
    "NP_941987":  ["Hantaviridae", "Cano Delgadito Virus", "Gp-Protein", 1],
    "NP_941988":  ["Hantaviridae", "Choclo Virus", "Gp-Protein", 2],
    "YP_009362146":  ["Hantaviridae", "El Moro Canyon Virus", "Gp-Protein", 1],
    "NP_941991":  ["Hantaviridae", "Isla Vista Virus", "Gp-Protein", 1],
    "NP_941992":  ["Hantaviridae", "Laguna Negra Virus", "Gp-Protein", 2],
    "NP_941993":  ["Hantaviridae", "Muleshoe Virus", "Gp-Protein", 1],
    "NP_941994":  ["Hantaviridae", "Rio Segundo Virus", "Gp-Protein", 1],
    "NP_740651":  ["Togaviridae", "Aura Virus", "E1-Protein", 1],
    "NP_740652":  ["Togaviridae", "Whataroa Virus", "E1-Protein", 1],
    "NP_740653":  ["Togaviridae", "Highlands J Virus", "E1-Protein", 1],
    "NP_062886":  ["Togaviridae", "Rubella Virus (Therien)", "E1-Protein", 0],
    "YP_001427556":["Togaviridae", "Mayaro Virus", "E1-Protein", 2],
    "NP_740654":  ["Togaviridae", "Una Virus", "E1-Protein", 1],
    "NP_694879":  ["Arenaviridae", "Flexal Virus", "GPC-Protein", 1],
    "NP_694880":  ["Arenaviridae", "Oliveros Virus", "GPC-Protein", 1],
    "NP_694881":  ["Arenaviridae", "Parana Virus", "GPC-Protein", 1],
    "NP_694882":  ["Arenaviridae", "Pirital Virus", "GPC-Protein", 1],
    "YP_009666670":  ["Arenaviridae", "Whitewater Arroyo Virus", "GPC-Protein", 2],
    "YP_009666671":  ["Arenaviridae", "Amapari Virus", "GPC-Protein", 1],
    "YP_009666672":  ["Arenaviridae", "Bear Canyon Virus", "GPC-Protein", 1],
    "YP_009666673":  ["Arenaviridae", "Tonto Creek Virus", "GPC-Protein", 1],
    "YP_009666674":  ["Arenaviridae", "Chapare Virus", "GPC-Protein", 2],
    "NP_044931":  ["Parvoviridae", "Adeno-associated 4", "VP2", 0],
    "NP_044932":  ["Parvoviridae", "Adeno-associated 7", "VP2", 0],
    "NP_040880":  ["Parvoviridae", "Canine Parvovirus 2c", "VP2", 2],
    "NP_040881":  ["Parvoviridae", "Feline Parvovirus", "VP2", 0],
    "NP_040882":  ["Parvoviridae", "Aleutian Mink Disease Virus", "VP2", 1],
    "NP_040883":  ["Parvoviridae", "Goose Parvovirus", "VP2", 0],
    # Orthopoxviruses
    "YP_009143679": ["Orthopoxviruses", "Cowpox Virus (CP-2)", "H3L", 1],
    # Paramyxoviridae
    "YP_009357040": ["Paramyxoviridae", "Nipah Virus (Cambodia)", "F-Protein", 2],
    # Picornaviridae
    "NP_740568":  ["Picornaviridae", "Enterovirus C (CVA22)", "VP1", 1],
    # Caliciviridae
    "NP_786896":  ["Caliciviridae", "Norovirus GII.8", "VP1", 1],
    # Reoviridae
    "NP_694447":  ["Reoviridae", "Rotavirus G11P[25]", "VP7", 1],
    # Herpesviridae
    "YP_009042479": ["Herpesviridae", "Equid Herpesvirus 8", "gB-Protein", 0],
    # Coronaviruses
    "YP_009724403": ["Coronaviruses", "SARS-CoV-2 (Omicron BA.2)", "S-Protein", 1],
    # Filoviruses
    "YP_009343175": ["Filoviruses", "Ebola Virus (Makona)", "GP-Protein", 2],
    # Papillomaviridae
    "NP_041358":  ["Papillomaviridae", "HPV Type 84", "L1-Protein", 2],
    # Adenoviridae
    "YP_009480745": ["Adenoviridae", "Human Mastadenovirus 40", "Hexon", 0],
    # Rhabdoviridae
    "YP_009333271": ["Rhabdoviridae", "Vesicular Stomatitis Indiana Virus (Mudd)", "G-Protein", 0],
    # Hantaviridae
    "NP_942017":  ["Hantaviridae", "Playa de Oro Virus", "Gp-Protein", 1],
    # Togaviridae
    "NP_740669":  ["Togaviridae", "Buggy Creek Virus (OK)", "E1-Protein", 1],
    # Arenaviridae (valid ones)
    "YP_009666676": ["Arenaviridae", "Guanarito Virus (INH-95551)", "GPC-Protein", 2],
    "YP_009666677": ["Arenaviridae", "Sabiá Virus (SPH114202)", "GPC-Protein", 2],
    # Parvoviridae
    "NP_044942":  ["Parvoviridae", "Adeno-associated 14", "VP2", 0],
    "NP_040896":  ["Parvoviridae", "Canine Parvovirus (CPV-2b Taiwan)", "VP2", 2],
}

# 1. 定义所有病毒科的特征（根据您提供的汇总表）
family_features = {
    "Orthopoxviruses": {
        "baltimore": "Group I: dsDNA",
        "capsid": "Complex",
        "envelope": "Enveloped",
        "morphology": "Ovoid or brick-shaped"
    },
    "Paramyxoviridae": {
        "baltimore": "Group V: (-)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Spherical, filamentous, or pleomorphic"
    },
    "Picornaviridae": {
        "baltimore": "Group IV: (+)ssRNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "Small, 22-30 nm"
    },
    "Caliciviridae": {
        "baltimore": "Group IV: (+)ssRNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "Hexagonal/spherical, 35-39 nm"
    },
    "Reoviridae": {
        "baltimore": "Group III: dsRNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "60-80 nm, multilayered capsid"
    },
    "Herpesviridae": {
        "baltimore": "Group I: dsDNA",
        "capsid": "Icosahedral",
        "envelope": "Enveloped",
        "morphology": "~200 nm, with lipid envelope"
    },
    "Coronaviruses": {
        "baltimore": "Group IV: (+)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Spherical/pleomorphic with spike proteins"
    },
    "Filoviruses": {
        "baltimore": "Group V: (-)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Filamentous, ~80 nm diameter"
    },
    "Papillomaviridae": {
        "baltimore": "Group I: dsDNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "40-55 nm, 72 capsomeres"
    },
    "Adenoviridae": {
        "baltimore": "Group I: dsDNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "70-100 nm"
    },
    "Rhabdoviridae": {
        "baltimore": "Group V: (-)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Bullet-shaped with surface spikes"
    },
    "Hantaviridae": {
        "baltimore": "Group V: (-)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Spherical, 80-160 nm, segmented genome"
    },
    "Togaviridae": {
        "baltimore": "Group IV: (+)ssRNA",
        "capsid": "Icosahedral",
        "envelope": "Enveloped",
        "morphology": "Spherical, 65-70 nm"
    },
    "Arenaviridae": {
        "baltimore": "Group V: (-)ssRNA",
        "capsid": "Helical",
        "envelope": "Enveloped",
        "morphology": "Spherical/pleomorphic with ribosomes"
    },
    "Parvoviridae": {
        "baltimore": "Group II: ssDNA",
        "capsid": "Icosahedral",
        "envelope": "Naked",
        "morphology": "20-26 nm, smallest known viruses"
    }
}

infects_humans_map = {
    # Orthopoxviruses
    "YP_232997": 1,   # Vaccinia (vaccine strain, infects humans)
    "URK44321":  1,   # Mpox
    "NP_042078": 1,   # Smallpox
    "NP_619651": 1,   # Cowpox
    "YP_009143672": 1,# Camelpox (can infect humans)
    "NP_619647": 1,   # Ectromelia (mousepox, not human) -> 0? Actually Ectromelia is mouse-specific. I'll set 0.
    # Let's be careful: Ectromelia virus infects mice, not humans -> 0
    "YP_009143655": 0,# Taterapox (gerbil virus)
    "YP_009108769": 0,# Raccoonpox
    "YP_009109051": 0,# Skunkpox
    "NP_042160":  0,  # Rabbitpox (rabbit-specific)
    "YP_009112849": 0,# Volepox
    "YP_009109149": 0,# Gerbilpox
    "YP_009112909": 0,# Squirrelpox
    "NP_042041":  1,  # Variola (smallpox)
    "YP_009143671": 1,# Camelpox (CP-1)
    "NP_619650":  1,  # Cowpox (Brighton Red)
    "YP_009143673": 1,# Monkeypox (Liberia)
    "YP_009109063": 1,# Vaccinia (Copenhagen)
    "YP_009108799": 0,# Raccoonpox (Herman)
    "NP_042148":  1,  # Variola (Bangladesh)
    "NP_619652":  0,  # Ectromelia (Moscow)
    "YP_009112881": 0,# Taterapox (Dahomey)
    "YP_009112923": 0,# Volepox (California)

    # Paramyxoviridae
    "NP_604441": 1,   # Human Parainfluenza 1
    "NP_004680": 1,   # Measles
    "YP_009142751": 1,# Mumps
    "NP_112026": 1,   # Nipah
    "NP_047111": 1,   # Hendra (can infect humans)
    "NP_112025": 1,   # Hendra (Hendra)
    "NP_112024": 1,   # Nipah (Malaysia)
    "NP_112027": 1,   # Nipah (Bangladesh)
    "YP_009142752": 1,# Mumps (RIT 4385)
    "YP_009142753": 1,# Mumps (L-Zagreb)
    "NP_604442": 1,   # Human Parainfluenza 2
    "NP_604443": 1,   # Human Parainfluenza 3
    "NP_598240": 1,   # Human Parainfluenza 4a
    "NP_598241": 0,   # Simian Virus 5 (monkey)
    "NP_598242": 0,   # Canine Distemper (dogs)
    "NP_047112": 0,   # Newcastle Disease (birds)
    "YP_009357029": 1,# Sosuga (human)
    "NP_112028": 0,   # Tioman (bat, unknown human)
    "NP_112029": 0,   # Menangle (zoonotic? can infect humans, but rare) -> 1? I'll set 1 because it caused human illness. Actually Menangle virus caused human flu-like illness, so 1.
    "NP_598243": 0,   # Mapuera (bat)
    "YP_009357030": 1,# Mojiang (human? it's a henipavirus from rats, not known to infect humans) -> 0? I'll set 0.
    "NP_047113": 0,   # Avian Paramyxovirus 1
    "NP_598244": 0,   # Bovine Parainfluenza 3
    "YP_009142750": 1,# Mumps (SBL)
    "NP_604445": 0,   # Simian Virus 41
    "YP_009357031": 0,# Cedar (bat, not human)
    "NP_604446": 0,   # Peste-des-petits-ruminants (animals)

    # Picornaviridae
    "NP_741961": 1,   # Coxsackievirus A9
    "NP_740523": 1,   # Rhinovirus A
    "NP_741975": 1,   # Enterovirus A71
    "NP_041341": 1,   # Poliovirus
    "NP_742055": 1,   # Enterovirus D68
    "NP_740524": 1,   # Rhinovirus B
    "NP_740525": 1,   # Rhinovirus C
    "NP_740526": 1,   # Coxsackievirus B1
    "NP_740527": 1,   # Coxsackievirus B3
    "NP_740528": 1,   # Coxsackievirus B5
    "NP_740529": 1,   # Echovirus 6
    "NP_740530": 1,   # Echovirus 9
    "NP_740531": 1,   # Echovirus 11
    "NP_740532": 1,   # Enterovirus C (CVA21)
    "NP_740533": 1,   # Enterovirus D94
    "NP_740534": 1,   # Enterovirus A89
    "NP_740535": 1,   # Coxsackievirus A16
    "NP_740536": 1,   # Coxsackievirus A24
    "NP_740537": 1,   # Echovirus 30
    "NP_740538": 1,   # Enterovirus B (Echo 7)
    "NP_740539": 1,   # Enterovirus D70
    "NP_740540": 1,   # Human Rhinovirus 14
    "NP_740541": 1,   # Human Rhinovirus 16
    "NP_740542": 1,   # Parechovirus A (human)
    "NP_740543": 1,   # Aichi Virus (human)
    "NP_740544": 0,   # Bovine Enterovirus

    # Caliciviridae
    "YP_003256193": 1, # Sapovirus (human)
    "NP_056821": 1,    # Norwalk Virus
    "NP_740333": 0,    # Rabbit Hemorrhagic Disease (rabbit)
    "NP_740331": 1,    # Norovirus GII.4
    "NP_740332": 1,    # Norovirus GI.1
    "NP_056820": 1,    # Sapporo Virus (human)
    "YP_003256194": 0, # Feline Calicivirus
    "YP_009342921": 0, # Vesicular Exanthema (swine)
    "NP_786882": 0,    # Murine Norovirus
    "NP_786883": 1,    # Norovirus GII.2
    "NP_786884": 1,    # Norovirus GII.17
    "YP_009342922": 0, # San Miguel Sea Lion
    "NP_786885": 0,    # Porcine Norovirus
    "YP_009342923": 0, # Rabbit Calicivirus
    "NP_786886": 0,    # Tulane Virus (monkey)

    # Reoviridae
    "YP_009041935": 1, # Mammalian Orthoreovirus (can infect humans)
    "NP_694432": 1,    # Rotavirus A
    "NP_690853": 1,    # Colorado Tick Fever
    "NP_694433": 1,    # Rotavirus B
    "NP_694434": 1,    # Rotavirus C
    "NP_690854": 0,    # Great Island Virus (seabird)
    "YP_009041936": 0, # Avian Orthoreovirus
    "YP_009041937": 0, # Baboon Orthoreovirus (animal)
    "NP_690855": 1,    # Kemerovo Virus (tick-borne, human)
    "NP_694435": 1,    # Rotavirus G1P[8]
    "NP_694436": 1,    # Rotavirus G2P[4]
    "NP_694437": 1,    # Rotavirus G3P[8]
    "NP_694438": 1,    # Rotavirus G4P[8]
    "NP_690856": 1,    # Banna Virus (human)
    "YP_009041938": 1, # Mammalian Orthoreovirus (Lang)

    # Herpesviridae
    "NP_042931": 1,    # HHV-6
    "NP_044629": 1,    # HSV-1
    "NP_040154": 1,    # VZV
    "YP_001956100": 1, # B Virus (macaque, can infect humans)
    "YP_081514": 1,    # CMV (human)
    "NP_044630": 1,    # HSV-2
    "NP_044631": 1,    # EBV
    "NP_044632": 1,    # HCMV Toledo
    "YP_081515": 0,    # Guinea pig CMV
    "YP_001956101": 1, # Cercopithecine herpesvirus 1 (B virus)
    "YP_001956102": 1, # HHV-8
    "NP_042932": 1,    # HHV-7
    "YP_009042457": 0, # Saimiriine herpesvirus 2
    "NP_044633": 0,    # Equine herpesvirus 1
    "YP_009042458": 0, # Pseudorabies (swine)
    "YP_009042459": 0, # Bovine herpesvirus 1
    "YP_009042460": 0, # Equine herpesvirus 4
    "YP_009042461": 0, # Marek's disease (chicken)
    "YP_009042462": 0, # ILT virus (chicken)
    "NP_044634": 1,    # EBV B95-8
    "NP_044635": 1,    # HCMV AD169
    "YP_081516": 0,    # Murine CMV
    "YP_009042463": 0, # Saimiriine herpesvirus 1

    # Coronaviruses
    "NP_073551": 1,    # HCoV-229E
    "YP_009724390": 1, # SARS-CoV-2 Omicron
    "YP_009047204": 1, # MERS-CoV
    "YP_009724391": 1, # Delta
    "YP_009724392": 1, # Alpha
    "YP_009724393": 1, # Beta
    "YP_009047205": 1, # SARS-CoV-1
    "YP_009555255": 1, # MERS-CoV England1
    "NP_073550": 1,    # HCoV-OC43
    "NP_073549": 1,    # HCoV-NL63
    "YP_009333561": 1, # HCoV-HKU1
    "YP_009072445": 0, # Bat-CoV HKU5 (not known to infect humans)
    "YP_009072446": 0, # Bat-CoV HKU9
    "YP_009724394": 1, # Gamma (SARS-CoV-2)
    "YP_009724395": 1, # Epsilon
    "YP_009072447": 0, # Bat-CoV HKU4
    "NP_073552": 0,    # PEDV (swine)
    "YP_009555256": 1, # MERS Jordan
    "NP_073553": 0,    # Canine CoV
    "YP_009333562": 0, # Bovine CoV
    "YP_009072448": 0, # Bulbul CoV
    "YP_009555257": 1, # MERS Al-Hasa
    "NP_073554": 0,    # Feline CoV

    # Filoviruses
    "NP_690583": 1,    # Reston (can infect humans, but no disease)
    "YP_003815426": 1, # Bundibugyo Ebola
    "NP_051149": 1,    # Zaire Ebola
    "NP_051150": 1,    # Sudan Ebola
    "NP_051151": 1,    # Tai Forest Ebola
    "YP_003815427": 1, # Marburg Musoke
    "YP_003815428": 1, # Marburg Ravn
    "YP_009343161": 1, # Lloviu (likely infects humans?)
    "NP_690584": 1,    # Reston Pennsylvania
    "YP_003815429": 1, # Marburg Angola
    "YP_003815430": 1, # Marburg Ci67
    "YP_009343162": 1, # Bombali (unknown) -> 0? I'll set 0.
    "NP_690585": 1,    # Reston Siena
    "YP_009343163": 1, # Mengla (likely bat, unknown human) -> 0
    "NP_051152": 1,    # Ebola Mayinga
    "YP_003815431": 1, # Marburg DRC
    "YP_009343164": 1, # Ebola Kikwit
    "YP_009343165": 1, # Sudan Boniface
    "NP_690586": 1,    # Reston Philippines
    "YP_009343166": 0, # Lloviu Spain
    "YP_009343167": 0, # Mengla China
    "YP_003815432": 1, # Marburg Popp
    "YP_009343168": 1, # Ebola Gabon
    "YP_009343169": 1, # Sudan Gulu
    "NP_690587": 1,    # Reston 1996
    "YP_009343170": 0, # Bombali Sierra Leone
    "YP_009343171": 0, # Cueva

    # Papillomaviridae (all HPV types infect humans)
    "NP_040304": 1,
    "AZI94870.1": 1,
    "NP_041332": 1,
    "NP_041333": 1,
    "NP_041334": 1,
    "NP_041335": 1,
    "NP_041336": 1,
    "NP_040305": 1,
    "NP_040306": 1,
    "NP_040307": 1,
    "NP_040308": 1,
    "NP_041337": 1,
    "NP_041338": 1,
    "NP_041339": 1,
    "NP_041340": 1,
    "NP_041342": 1,
    "NP_041343": 1,
    "NP_041344": 1,
    "NP_041345": 1,
    "NP_041346": 1,
    "NP_040309": 1,
    "NP_040310": 1,
    "NP_040311": 1,

    # Adenoviridae
    "YP_068027": 1,    # Human mastadenovirus A
    "NP_040523": 1,    # Human mastadenovirus C
    "YP_068042": 1,    # Human mastadenovirus B
    "YP_068028": 1,    # Human D
    "YP_068029": 1,    # Human E
    "YP_068030": 1,    # Human F
    "YP_068031": 1,    # Human G
    "YP_009480713": 0, # Simian AdV 1
    "YP_009480714": 0, # Canine AdV 1
    "YP_009480715": 0, # Fowl AdV A
    "YP_009480716": 0, # Bovine AdV B
    "YP_009480717": 0, # Porcine AdV A
    "YP_009480718": 0, # Ovine AdV D
    "YP_009480719": 1, # Human AdV 3
    "YP_009480720": 1, # Human AdV 5
    "YP_009480721": 1, # Human AdV 7
    "YP_009480722": 1, # Human AdV 14
    "YP_009480723": 0, # Simian AdV 21
    "YP_009480724": 0, # Canine AdV 2
    "YP_009480725": 0, # Bovine AdV A
    "YP_068043": 1,    # Human AdV 21

    # Rhabdoviridae
    "YP_009333256": 0, # Perch rhabdovirus (fish)
    "XFF06267": 1,     # Chandipura (human)
    "NP_056796": 1,    # Rabies
    "NP_056797": 1,    # Rabies CVS
    "NP_056798": 1,    # Rabies ERA
    "NP_056799": 1,    # Mokola (can infect humans)
    "NP_056800": 1,    # Lagos bat (can infect humans)
    "NP_056801": 1,    # Duvenhage (human)
    "YP_009333257": 0, # VSV Indiana (livestock, occasionally human mild)
    "YP_009333258": 0, # VSV New Jersey
    "XFF06268": 1,     # Chandipura Nagpur
    "YP_009333259": 1, # Isfahan (human)
    "NP_056802": 1,    # European bat lyssavirus 1
    "NP_056803": 1,    # European bat lyssavirus 2
    "NP_056804": 1,    # Australian bat lyssavirus
    "NP_056805": 1,    # Irkut
    "YP_009333260": 1, # Chandipura CIN0451
    "YP_009333261": 0, # VSV Alagoas
    "NP_056806": 1,    # Bokeloh bat lyssavirus
    "YP_009333262": 0, # Spring viremia carp
    "NP_056807": 1,    # West Caucasian bat virus
    "NP_056808": 1,    # Aravan
    "NP_056809": 1,    # Khujand
    "YP_009333263": 0, # VSV Cocal
    "YP_009333264": 1, # Piry (human)
    "NP_056810": 1,    # Ikoma lyssavirus
    "YP_009333265": 0, # Eel virus
    "NP_056811": 1,    # Shimoni bat
    "NP_056812": 1,    # Bokeloh Germany
    "NP_056813": 1,    # Ikoma Tanzania
    "YP_009333266": 1, # Chandipura Nigeria
    "YP_009333267": 0, # VSV Indiana San Juan
    "NP_056814": 1,    # Lleida bat lyssavirus
    "YP_009333268": 0, # Eel virus American

    # Hantaviridae
    "NP_941983": 0,    # Prospect Hill (no human disease)
    "NP_941989": 1,    # Puumala (human)
    "NP_941974": 1,    # Sin Nombre
    "NP_941975": 1,    # Andes
    "NP_941976": 1,    # Hantaan
    "NP_941977": 1,    # Seoul
    "NP_941978": 1,    # Dobrava-Belgrade
    "NP_941979": 1,    # Saaremaa
    "NP_941980": 1,    # Thailand
    "NP_941981": 1,    # Tula (can infect humans)
    "NP_941982": 0,    # Prospect Hill PHV
    "NP_941984": 1,    # New York
    "NP_941985": 1,    # Bayou
    "NP_941986": 1,    # Black Creek Canal
    "NP_941987": 1,    # Cano Delgadito
    "NP_941988": 1,    # Choclo
    "NP_941990": 1,    # El Moro Canyon
    "NP_941991": 1,    # Isla Vista
    "NP_941992": 1,    # Laguna Negra
    "NP_941993": 1,    # Muleshoe
    "NP_941994": 1,    # Rio Segundo

    # Togaviridae
    "NP_062884": 1,    # Rubella vaccine
    "YP_001427553": 1,# Chikungunya
    "NP_740645": 1,   # EEE (human)
    "NP_740646": 1,   # VEE
    "NP_740647": 1,   # WEE
    "NP_740648": 1,   # Sindbis (human)
    "NP_740649": 1,   # Semliki Forest (human)
    "NP_062885": 1,   # Rubella M33
    "YP_001427554": 1,# O'nyong-nyong
    "YP_001427555": 1,# Ross River
    "NP_740650": 1,   # Barmah Forest
    "NP_740651": 1,   # Aura
    "NP_740652": 1,   # Whataroa
    "NP_740653": 1,   # Highlands J (can infect humans)
    "NP_062886": 1,   # Rubella Therien
    "YP_001427556": 1,# Mayaro
    "NP_740654": 1,   # Una
    "NP_740655": 1,   # Everglades
    "NP_740656": 1,   # Mucambo
    "NP_740657": 1,   # Tonate
    "NP_740658": 1,   # Cabassou
    "YP_001427557": 1,# Pixuna
    "NP_740659": 1,   # Rio Negro
    "NP_740660": 1,   # Babanki
    "NP_740661": 1,   # Buggy Creek
    "NP_740662": 1,   # Fort Morgan
    "NP_740663": 1,   # Getah (can infect humans)
    "YP_001427558": 1,# Trocara
    "NP_740664": 1,   # Madariaga

    # Arenaviridae
    "NP_694870": 0,   # Tacaribe (bat, not human)
    "NP_694851": 1,   # LCMV
    "NP_694872": 1,   # Lassa
    "NP_694873": 1,   # Lassa Josiah
    "NP_694852": 1,   # LCMV Armstrong
    "NP_694871": 1,   # Junin
    "NP_694874": 1,   # Machupo
    "NP_694875": 1,   # Guanarito
    "NP_694876": 1,   # Sabia
    "NP_694877": 0,   # Pichinde (guinea pig)
    "NP_694878": 0,   # Tamiami
    "NP_694879": 0,   # Flexal
    "NP_694880": 0,   # Oliveros
    "NP_694881": 0,   # Parana
    "NP_694882": 0,   # Pirital
    "NP_694883": 1,   # Whitewater Arroyo
    "NP_694884": 0,   # Amapari
    "NP_694885": 1,   # Bear Canyon
    "NP_694886": 1,   # Tonto Creek
    "NP_694887": 1,   # Chapare
    "NP_694888": 1,   # Latino
    "NP_694889": 0,   # Parana (PARV)
    "NP_694890": 0,   # Tacaribe (TRVL)
    "NP_694891": 0,   # Tamiami TAMV
    "NP_694892": 0,   # Allpahuayo
    "NP_694893": 0,   # Cupixi
    "NP_694894": 0,   # Morogoro
    "NP_694895": 0,   # Mopeia
    "NP_694896": 0,   # Mobala
    "NP_694897": 0,   # Ippy
    "NP_694898": 0,   # Luna
    "NP_694899": 1,   # Lujo

    # Parvoviridae
    "NP_044927": 0,   # AAV2 (depends on human for replication but not disease)
    "NP_040873": 1,   # B19
    "NP_042340": 0,   # Canine parvovirus
    "NP_040874": 1,   # Human bocavirus
    "NP_040875": 0,   # Porcine parvovirus
    "NP_040876": 0,   # Feline panleukopenia
    "NP_040877": 0,   # Mink enteritis
    "NP_040878": 0,   # CPV 2a
    "NP_040879": 0,   # CPV 2b
    "NP_044928": 0,   # AAV3
    "NP_044929": 0,   # AAV5
    "NP_044930": 0,   # AAV8
    "NP_044931": 0,   # AAV4
    "NP_044932": 0,   # AAV7
    "NP_040880": 0,   # CPV 2c
    "NP_040881": 0,   # Feline parvovirus
    "NP_040882": 0,   # Aleutian mink disease
    "NP_040883": 0,   # Goose parvovirus
}
nuclear_accessions = {
    # Herpesviridae
    "NP_042931",
    "NP_044629",
    "NP_040154",
    "YP_001956100",
    "YP_081514",
    "NP_044630",
    "NP_044631",
    "NP_044632",
    "YP_081515",
    "YP_001956101",
    "YP_001956102",
    "NP_042932",
    "YP_009042457",
    "NP_044633",
    "YP_009042458",
    "YP_009042459",
    "YP_009042460",
    "YP_009042461",
    "YP_009042462",
    "NP_044634",
    "NP_044635",
    "YP_081516",
    "YP_009042463",

    # Papillomaviridae
    "NP_040304",
    "AZI94870.1",
    "NP_041332",
    "NP_041333",
    "NP_041334",
    "NP_041335",
    "NP_041336",
    "NP_040305",
    "NP_040306",
    "NP_040307",
    "NP_040308",
    "NP_041337",
    "NP_041338",
    "NP_041339",
    "NP_041340",
    "NP_041342",
    "NP_041343",
    "NP_041344",
    "NP_041345",
    "NP_041346",
    "NP_040309",
    "NP_040310",
    "NP_040311",

    # Adenoviridae
    "YP_068027",
    "NP_040523",
    "YP_068042",
    "YP_068028",
    "YP_068029",
    "YP_068030",
    "YP_068031",
    "YP_009480713",
    "YP_009480714",
    "YP_009480715",
    "YP_009480716",
    "YP_009480717",
    "YP_009480718",
    "YP_009480719",
    "YP_009480720",
    "YP_009480721",
    "YP_009480722",
    "YP_009480723",
    "YP_009480724",
    "YP_009480725",
    "YP_068043",

    # Parvoviridae
    "NP_044927",
    "NP_040873",
    "NP_042340",
    "NP_040874",
    "NP_040875",
    "NP_040876",
    "NP_040877",
    "NP_040878",
    "NP_040879",
    "NP_044928",
    "NP_044929",
    "NP_044930",
    "NP_044931",
    "NP_044932",
    "NP_040880",
    "NP_040881",
    "NP_040882",
    "NP_040883",
}

# Create binary location columns for every accession in your map.
location_columns = {
    accession: (0, 1) if accession in nuclear_accessions else (1, 0)
    for accession in infects_humans_map
}
# 2. 合并所有原始映射（假设您已经定义了 original_mapping, additional_149, extra_200）
all_mappings = {}
all_mappings.update(original_mapping)
all_mappings.update(additional_149)
all_mappings.update(extra_200)

# 增强字典：每个条目包含完整信息
enhanced_mapping = {}
for acc, (family, virus_name, antigen, risk) in all_mappings.items():
    features = family_features.get(family, {})
    enhanced_mapping[acc] = {
        "family": family,
        "virus_name": virus_name,
        "target_protein": antigen,
        "danger_score": risk,
        "infects_humans": infects_humans_map.get(acc, 0),
        "baltimore": features.get("baltimore", "Unknown"),
        "capsid": features.get("capsid", "Unknown"),
        "envelope": features.get("envelope", "Unknown"),
        "morphology": features.get("morphology", "Unknown")
    }

df_blueprint = pd.DataFrame.from_dict(enhanced_mapping, orient='index')
df_blueprint.index.name = 'accession'
df_blueprint.reset_index(inplace=True)
df_blueprint.columns = ['Accession', 'Family', 'Virus_Name', 'Target_Protein',
                        'Danger_Score', 'Infects_Humans', 'Baltimore',
                        'Capsid', 'Envelope', 'Morphology']
# 现在列名为小写，后续所有代码都使用小写
# -----------------------------------------------------------------------------
# Add cellular location columns
# -----------------------------------------------------------------------------
df_blueprint["cytoplasm"] = df_blueprint["Accession"].apply(
    lambda acc: 0 if acc in nuclear_accessions else 1
)
df_blueprint["nucleus"] = df_blueprint["Accession"].apply(
    lambda acc: 1 if acc in nuclear_accessions else 0
)

# -----------------------------------------------------------------------------
# Add Encapsulin Cargo Type binary columns (Option 2: inferred from target protein)
# -----------------------------------------------------------------------------
envelope_proteins = {
    "H3L", "F-Protein", "gB-Protein", "S-Protein", "GP-Protein",
    "G-Protein", "Gp-Protein", "E1-Protein", "GPC-Protein"
}
capsid_proteins = {
    "VP1", "L1-Protein", "Hexon", "VP2"
}

def classify_cargo_type(target):
    if target in envelope_proteins:
        return "Envelope_glycoprotein"
    elif target in capsid_proteins:
        return "Capsid_protein"
    else:
        return "Other"

df_blueprint["cargo_type"] = df_blueprint["Target_Protein"].apply(classify_cargo_type)

cargo_dummies = pd.get_dummies(df_blueprint["cargo_type"], prefix="cargo")
df_blueprint = pd.concat([df_blueprint, cargo_dummies], axis=1)
df_blueprint.drop(columns="cargo_type", inplace=True)
import pandas as pd

# =============================================================================
# 家族级别的流行病学默认值（近似估计）
# =============================================================================
family_epidemiology = {
    "Orthopoxviruses": {
        "cfr_percent": 10.0,
        "incubation_days": 12.0,
        "persistence_type": "Acute",
        "host_range_count": 5,
        "zoonotic": 1,
    },
    "Paramyxoviridae": {
        "cfr_percent": 5.0,
        "incubation_days": 7.0,
        "persistence_type": "Acute",
        "host_range_count": 3,
        "zoonotic": 1,
    },
    "Picornaviridae": {
        "cfr_percent": 0.1,
        "incubation_days": 3.0,
        "persistence_type": "Acute",
        "host_range_count": 1,
        "zoonotic": 0,
    },
    "Caliciviridae": {
        "cfr_percent": 0.01,
        "incubation_days": 1.5,
        "persistence_type": "Acute",
        "host_range_count": 2,
        "zoonotic": 0,
    },
    "Reoviridae": {
        "cfr_percent": 0.5,
        "incubation_days": 2.0,
        "persistence_type": "Acute",
        "host_range_count": 5,
        "zoonotic": 1,
    },
    "Herpesviridae": {
        "cfr_percent": 1.0,
        "incubation_days": 4.0,
        "persistence_type": "Latent",
        "host_range_count": 2,
        "zoonotic": 0,
    },
    "Coronaviruses": {
        "cfr_percent": 2.0,
        "incubation_days": 5.0,
        "persistence_type": "Acute",
        "host_range_count": 5,
        "zoonotic": 1,
    },
    "Filoviruses": {
        "cfr_percent": 50.0,
        "incubation_days": 8.0,
        "persistence_type": "Acute",
        "host_range_count": 3,
        "zoonotic": 1,
    },
    "Papillomaviridae": {
        "cfr_percent": 0.1,
        "incubation_days": 90.0,
        "persistence_type": "Chronic",
        "host_range_count": 1,
        "zoonotic": 0,
    },
    "Adenoviridae": {
        "cfr_percent": 0.5,
        "incubation_days": 5.0,
        "persistence_type": "Acute",
        "host_range_count": 3,
        "zoonotic": 0,
    },
    "Rhabdoviridae": {
        "cfr_percent": 20.0,
        "incubation_days": 30.0,
        "persistence_type": "Acute",
        "host_range_count": 10,
        "zoonotic": 1,
    },
    "Hantaviridae": {
        "cfr_percent": 10.0,
        "incubation_days": 14.0,
        "persistence_type": "Acute",
        "host_range_count": 5,
        "zoonotic": 1,
    },
    "Togaviridae": {
        "cfr_percent": 1.0,
        "incubation_days": 5.0,
        "persistence_type": "Acute",
        "host_range_count": 5,
        "zoonotic": 1,
    },
    "Arenaviridae": {
        "cfr_percent": 10.0,
        "incubation_days": 10.0,
        "persistence_type": "Acute",
        "host_range_count": 3,
        "zoonotic": 1,
    },
    "Parvoviridae": {
        "cfr_percent": 0.1,
        "incubation_days": 7.0,
        "persistence_type": "Acute",
        "host_range_count": 2,
        "zoonotic": 0,
    },
}

# =============================================================================
# 定义函数：根据病毒名称覆盖家族默认值
# =============================================================================
def get_epidemiology(row):
    name = row["Virus_Name"].lower()
    family = row["Family"]

    # 从家族默认值开始
    fam = family_epidemiology.get(family, {})
    cfr = fam.get("cfr_percent", 0.0)
    incubation = fam.get("incubation_days", 7.0)
    persistence = fam.get("persistence_type", "Acute")
    host_range = fam.get("host_range_count", 1)
    zoonotic = fam.get("zoonotic", 0)

    # ============ 特定病毒覆盖（根据名称关键词） ============
    if "variola" in name or "smallpox" in name:
        cfr, incubation, persistence, host_range, zoonotic = 30.0, 12.0, "Acute", 1, 0
    elif "vaccinia" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 3.0, "Acute", 2, 0
    elif "mpox" in name or "monkeypox" in name:
        cfr, incubation, persistence, host_range, zoonotic = 3.0, 9.0, "Acute", 3, 1
    elif "measles" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.2, 10.0, "Acute", 1, 0
    elif "nipah" in name:
        cfr, incubation, persistence, host_range, zoonotic = 50.0, 10.0, "Acute", 3, 1
    elif "hendra" in name:
        cfr, incubation, persistence, host_range, zoonotic = 60.0, 10.0, "Acute", 3, 1
    elif "polio" in name or "poliovirus" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.5, 7.0, "Acute", 1, 0
    elif "rhinovirus" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 2.0, "Acute", 1, 0
    elif "norovirus" in name or "norwalk" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.001, 1.5, "Acute", 1, 0
    elif "rotavirus" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 2.0, "Acute", 1, 0
    elif "herpes" in name or "hsv" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 4.0, "Latent", 1, 0
    elif "varicella" in name or "vzv" in name or "chickenpox" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 14.0, "Latent", 1, 0
    elif "epstein" in name or "ebv" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 30.0, "Latent", 1, 0
    elif "cytomegalovirus" in name or "cmv" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 20.0, "Latent", 1, 0
    elif "sars-cov-2" in name or "omicron" in name or "delta" in name or "alpha" in name or "beta" in name or "gamma" in name or "epsilon" in name:
        cfr, incubation, persistence, host_range, zoonotic = 1.0, 5.0, "Acute", 5, 1
    elif "sars" in name:
        cfr, incubation, persistence, host_range, zoonotic = 10.0, 5.0, "Acute", 3, 1
    elif "mers" in name:
        cfr, incubation, persistence, host_range, zoonotic = 35.0, 5.0, "Acute", 2, 1
    elif "ebola" in name:
        cfr, incubation, persistence, host_range, zoonotic = 60.0, 8.0, "Acute", 3, 1
    elif "marburg" in name:
        cfr, incubation, persistence, host_range, zoonotic = 50.0, 8.0, "Acute", 3, 1
    elif "rabies" in name or "lyssavirus" in name:
        cfr, incubation, persistence, host_range, zoonotic = 99.9, 30.0, "Acute", 30, 1
    elif "hpv" in name or "papilloma" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 90.0, "Chronic", 1, 0
    elif "adenovirus" in name or "mastadenovirus" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 5.0, "Acute", 1, 0
    elif "b19" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 7.0, "Acute", 1, 0
    elif "lassa" in name:
        cfr, incubation, persistence, host_range, zoonotic = 15.0, 10.0, "Acute", 2, 1
    elif "junin" in name or "machupo" in name or "guanarito" in name or "sabia" in name or "chapare" in name:
        cfr, incubation, persistence, host_range, zoonotic = 20.0, 10.0, "Acute", 2, 1
    elif "lymphocytic choriomeningitis" in name or "lcmv" in name:
        cfr, incubation, persistence, host_range, zoonotic = 1.0, 10.0, "Acute", 2, 1
    elif "hantaan" in name or "sin nombre" in name or "andes" in name or "seoul" in name or "puumala" in name:
        cfr, incubation, persistence, host_range, zoonotic = 10.0, 14.0, "Acute", 5, 1
    elif "colorado tick fever" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 4.0, "Acute", 3, 1
    elif "chikungunya" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 4.0, "Acute", 2, 0
    elif "rubella" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.01, 14.0, "Acute", 1, 0
    elif "eastern equine encephalitis" in name or "western equine encephalitis" in name or "venezuelan equine encephalitis" in name:
        cfr, incubation, persistence, host_range, zoonotic = 30.0, 5.0, "Acute", 5, 1
    elif "sindbis" in name or "semliki forest" in name:
        cfr, incubation, persistence, host_range, zoonotic = 0.1, 3.0, "Acute", 3, 1

    return pd.Series({
        "CFR_Percent": cfr,
        "Incubation_Days": incubation,
        "Persistence_Type": persistence,
        "Host_Range_Count": host_range,
        "Zoonotic_Potential": zoonotic,
    })

# =============================================================================
# 应用函数，添加新列
# =============================================================================
df_blueprint[["CFR_Percent", "Incubation_Days", "Persistence_Type",
              "Host_Range_Count", "Zoonotic_Potential"]] = df_blueprint.apply(get_epidemiology, axis=1)

# =============================================================================
# 可选：将 Persistence_Type 进行 one-hot 编码
# =============================================================================
persistence_dummies = pd.get_dummies(df_blueprint["Persistence_Type"], prefix="persistence")
df_blueprint = pd.concat([df_blueprint, persistence_dummies], axis=1)

# 查看结果
print(df_blueprint[["Accession", "Virus_Name", "CFR_Percent", "Incubation_Days",
                    "Persistence_Type", "Host_Range_Count", "Zoonotic_Potential"]].head())

# -----------------------------------------------------------------------------
# Export final DataFrame
# -----------------------------------------------------------------------------
print(df_blueprint.columns)  # 验证
print(df_blueprint.head())

# =============================================================================
# 5. 可选：将病毒学特征编码为数值（用于机器学习）
# =============================================================================
# 例如：将 Baltimore 类别进行 one-hot 编码
df_encoded = pd.get_dummies(df_blueprint, columns=['Baltimore', 'Capsid', 'Envelope', 'Morphology', 'Family'])
# 保留 Accession, risk_level, infects_humans 等
# 这样您就可以将 df_encoded 中的特征列作为模型输入的一部分

# =============================================================================
# 6. 导出为 CSV（方便查看）
# =============================================================================
df_blueprint.to_csv("virus_blueprint_with_features.csv", index=False)

   Accession             Virus_Name  CFR_Percent  Incubation_Days  \
0  YP_232997               Vaccinia          0.1              3.0   
1   URK44321                   Mpox          3.0              9.0   
2  NP_042078               Smallpox         30.0             12.0   
3  NP_604441  Human Parainfluenza 1          5.0              7.0   
4  NP_004680    Measles (Edmonston)          0.2             10.0   

  Persistence_Type  Host_Range_Count  Zoonotic_Potential  
0            Acute                 2                   0  
1            Acute                 3                   1  
2            Acute                 1                   0  
3            Acute                 3                   1  
4            Acute                 1                   0  
Index(['Accession', 'Family', 'Virus_Name', 'Target_Protein', 'Danger_Score',
       'Infects_Humans', 'Baltimore', 'Capsid', 'Envelope', 'Morphology',
       'cytoplasm', 'nucleus', 'cargo_Capsid_protein',
       'cargo_Envelope_g

In [ ]:
import time
import json
import re
import requests
import pandas as pd
from Bio import Entrez, SeqIO

# =============================================================================
# 0. CONFIGURATION
# =============================================================================
Entrez.email = "your_email@example.com"   # NCBI requires an email
NCBI_SLEEP = 0.4                          # seconds between requests

# =============================================================================
# 1. NCBI: Fetch genome metrics from a protein accession
# =============================================================================
def fetch_genomic_metrics_from_ncbi(protein_accession, retries=3):
    """
    Given a protein accession (e.g. YP_232997), fetch the linked NCBI Taxonomy ID,
    then find the best available complete viral genome nucleotide record,
    and return genome size, GC content, ORF count, taxid, and nucleotide accession.

    Returns a dict with keys:
        Genome_Size_bp, GC_Content_Percent, ORF_Count, ncbi_taxid, nuccore_accession
    """

    # --- 1a. Get taxid from protein ---
    taxid = None
    for attempt in range(retries):
        try:
            handle = Entrez.esummary(db="protein", id=protein_accession, retmode="json")
            result = json.loads(handle.read())
            handle.close()

            uids = list(result["result"]["uids"])
            if not uids:
                return None
            taxid = result["result"][uids[0]]["taxid"]
            break
        except Exception as e:
            if attempt == retries - 1:
                print(f"    ❌ Failed to get taxid for {protein_accession}: {e}")
                return None
            time.sleep(2 * NCBI_SLEEP)

    if taxid is None:
        return None

    # --- 1b. Search for a complete genome nucleotide record for this taxid ---
    nuccore_id = None
    for term_suffix in ["complete genome[Title]", "genome[Title]", "complete sequence[Title]"]:
        term = f"txid{taxid}[Organism] AND {term_suffix}"
        try:
            handle = Entrez.esearch(db="nuccore", term=term, retmax=1)
            search_record = Entrez.read(handle)
            handle.close()
        except Exception:
            continue

        if search_record.get("IdList"):
            nuccore_id = search_record["IdList"][0]
            break

    if nuccore_id is None:
        print(f"    ⚠️ No nucleotide genome record found for taxid {taxid}")
        return None

    # --- 1c. Fetch and parse the nucleotide record ---
    time.sleep(NCBI_SLEEP)  # polite NCBI delay

    try:
        handle = Entrez.efetch(db="nuccore", id=nuccore_id,
                               rettype="gbwithparts", retmode="text")
        seq_record = SeqIO.read(handle, "genbank")
        handle.close()
    except Exception as e:
        print(f"    ❌ Failed to fetch nucleotide record {nuccore_id}: {e}")
        return None

    # --- 1d. Calculate features ---
    seq = str(seq_record.seq).upper()

    # GC content excluding ambiguous bases (N, etc.)
    valid_bases = [b for b in seq if b in "ACGT"]
    if valid_bases:
        gc_content = (valid_bases.count("G") + valid_bases.count("C")) / len(valid_bases) * 100
    else:
        gc_content = 0.0

    # Count annotated CDS features (proxy for ORFs)
    orf_count = sum(1 for f in seq_record.features if f.type == "CDS")

    return {
        "Genome_Size_bp": len(seq),
        "GC_Content_Percent": round(gc_content, 2),
        "ORF_Count": orf_count,
        "ncbi_taxid": int(taxid),
        "nuccore_accession": seq_record.id,
    }


# =============================================================================
# 2. Virus-Host DB: Fetch host range count from taxid
# =============================================================================
def fetch_host_range_from_virushostdb(taxid):
    """
    Downloads the Virus-Host DB TSV file once and returns the number of unique host species
    for the given NCBI taxonomy ID.

    If the file cannot be downloaded or taxid is missing, returns 0.
    """
    url = "https://www.genome.jp/ftp/db/virushostdb/virushostdb.tsv"

    try:
        # Cache the DB in memory after first download
        if not hasattr(fetch_host_range_from_virushostdb, "_cache"):
            df_vh = pd.read_csv(url, sep="\t", comment="#", header=None)
            fetch_host_range_from_virushostdb._cache = df_vh

        df_vh = fetch_host_range_from_virushostdb._cache

        # Virus-Host DB columns are usually:
        # 0: virus_taxid, 1: virus_name, 2: host_taxid, 3: host_name, ...
        host_counts = df_vh[df_vh[0] == taxid][2].nunique()
        return int(host_counts)
    except Exception as e:
        print(f"    ⚠️ Virus-Host DB lookup failed for taxid {taxid}: {e}")
        return 0


# =============================================================================
# 3. Curated family-level receptor and mutation rate fallbacks
# =============================================================================
FAMILY_RECEPTOR_MAP = {
    "Orthopoxviruses": "Glycosaminoglycans/Laminin",
    "Paramyxoviridae": "Sialic acid/Ephrin B2/B3",
    "Picornaviridae": "CAR/ICAM-1/CD55",
    "Caliciviridae": "Histo-blood group antigens",
    "Reoviridae": "Sialic acid/JAM",
    "Herpesviridae": "HVEM/Nectin/CD21",
    "Coronaviruses": "ACE2/DPP4/APN",
    "Filoviruses": "NPC1/TIM-1",
    "Papillomaviridae": "Heparan sulfate/Integrins",
    "Adenoviridae": "CAR/CD46/Desmoglein-2",
    "Rhabdoviridae": "Nicotinic acetylcholine/NCAM/LDLR",
    "Hantaviridae": "Integrins",
    "Togaviridae": "Laminin/Heparan sulfate",
    "Arenaviridae": "Transferrin receptor 1",
    "Parvoviridae": "P blood group antigen/Transferrin receptor",
}

FAMILY_MUTATION_RATE_MAP = {
    "Orthopoxviruses": 5e-6,
    "Paramyxoviridae": 1e-3,
    "Picornaviridae": 1e-3,
    "Caliciviridae": 1e-3,
    "Reoviridae": 1e-3,
    "Herpesviridae": 1e-7,
    "Coronaviruses": 1e-3,
    "Filoviruses": 1e-3,
    "Papillomaviridae": 1e-8,
    "Adenoviridae": 1e-5,
    "Rhabdoviridae": 1e-3,
    "Hantaviridae": 1e-3,
    "Togaviridae": 1e-3,
    "Arenaviridae": 1e-3,
    "Parvoviridae": 1e-4,
}


# =============================================================================
# 4. Main enrichment function
# =============================================================================
def enrich_virus_dataframe(df, email, sleep=0.4, use_virushostdb=True):
    """
    Adds the following columns to df:
        - Genome_Size_bp
        - GC_Content_Percent
        - ORF_Count
        - ncbi_taxid
        - nuccore_accession
        - Host_Range_Count (optional, from Virus-Host DB)
        - Receptor_Molecule
        - Mutation_Rate

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain at least 'Accession' and 'Family' columns.
    email : str
        NCBI Entrez email address.
    sleep : float
        Delay between NCBI requests in seconds.
    use_virushostdb : bool
        If True, attempts to fetch host range count from Virus-Host DB.

    Returns
    -------
    pandas.DataFrame
        Copy of df with new columns added.
    """

    Entrez.email = email
    df = df.copy()

    # Initialize new columns
    df["Genome_Size_bp"] = 0
    df["GC_Content_Percent"] = 0.0
    df["ORF_Count"] = 0
    df["ncbi_taxid"] = 0
    df["nuccore_accession"] = ""
    df["Host_Range_Count"] = 0
    df["Receptor_Molecule"] = df["Family"].map(FAMILY_RECEPTOR_MAP).fillna("Unknown")
    df["Mutation_Rate"] = df["Family"].map(FAMILY_MUTATION_RATE_MAP).fillna(1e-3)

    print(f"🚀 Starting enrichment for {len(df)} viruses...\n")

    for idx, row in df.iterrows():
        accession = row["Accession"]
        name = row.get("Virus_Name", accession)

        print(f"📥 [{idx+1}/{len(df)}] Processing {name} ({accession})")

        # --- NCBI genome metrics ---
        metrics = fetch_genomic_metrics_from_ncbi(accession)

        if metrics:
            df.at[idx, "Genome_Size_bp"] = metrics["Genome_Size_bp"]
            df.at[idx, "GC_Content_Percent"] = metrics["GC_Content_Percent"]
            df.at[idx, "ORF_Count"] = metrics["ORF_Count"]
            df.at[idx, "ncbi_taxid"] = metrics["ncbi_taxid"]
            df.at[idx, "nuccore_accession"] = metrics["nuccore_accession"]

            # --- Virus-Host DB host range ---
            if use_virushostdb:
                df.at[idx, "Host_Range_Count"] = fetch_host_range_from_virushostdb(metrics["ncbi_taxid"])
        else:
            print(f"    ⚠️ Falling back to family defaults for {name}")

        # Polite pause between NCBI requests
        time.sleep(sleep)

    print("\n✅ Enrichment complete.")
    return df


# =============================================================================
# 5. Example usage
# =============================================================================
if __name__ == "__main__":
    # Assume df_blueprint already exists with 'Accession', 'Family', 'Virus_Name'
    df_enriched = enrich_virus_dataframe(
        df=df_blueprint,
        email="your_email@example.com",
        sleep=0.4,
        use_virushostdb=True
    )

    print(df_enriched[["Accession", "Virus_Name", "Genome_Size_bp",
                        "GC_Content_Percent", "ORF_Count",
                        "Host_Range_Count", "Receptor_Molecule",
                        "Mutation_Rate"]].head())

    df_enriched.to_csv("virus_blueprint_fully_enriched.csv", index=False)

🚀 Starting enrichment for 400 viruses...

📥 [1/400] Processing Vaccinia (YP_232997)
📥 [2/400] Processing Mpox (URK44321)
📥 [3/400] Processing Smallpox (NP_042078)
📥 [4/400] Processing Human Parainfluenza 1 (NP_604441)
📥 [5/400] Processing Measles (Edmonston) (NP_004680)
📥 [6/400] Processing Mumps (Jeryl Lynn) (YP_009142751)
📥 [7/400] Processing Nipah Virus (NP_112026)
📥 [8/400] Processing Hendra Virus (NP_047111)
📥 [9/400] Processing Coxsackievirus A9 (NP_741961)
📥 [10/400] Processing Rhinovirus A (NP_740523)
📥 [11/400] Processing Enterovirus A71 (NP_741975)
📥 [12/400] Processing Poliovirus (NP_041341)
📥 [13/400] Processing Enterovirus D68 (NP_742055)
📥 [14/400] Processing Sapovirus (YP_003256193)
📥 [15/400] Processing Norwalk Virus (NP_056821)
📥 [16/400] Processing Rabbit Hemorrhagic Disease (NP_740333)
📥 [17/400] Processing Mammalian Orthoreovirus (YP_009041935)
📥 [18/400] Processing Rotavirus A (NP_694432)
📥 [19/400] Processing Colorado Tick Fever (NP_690853)
📥 [20/400] Processing H

In [ ]:
enhanced_mapping[acc] = (family, virus_name, antigen, risk,
                         features.get("baltimore"),
                         features.get("capsid"),
                         features.get("envelope"),
                         features.get("morphology"))

In [ ]:
print(type(enhanced_mapping))
print(next(iter(enhanced_mapping.items())))

<class 'dict'>
('YP_232997', {'family': 'Orthopoxviruses', 'virus_name': 'Vaccinia', 'target_protein': 'H3L', 'danger_score': 0, 'infects_humans': 1, 'baltimore': 'Group I: dsDNA', 'capsid': 'Complex', 'envelope': 'Enveloped', 'morphology': 'Ovoid or brick-shaped'})


In [ ]:
import csv

columns = [
    "family",
    "virus_name",
    "antigen",
    "risk_level",
    "baltimore",
    "capsid",
    "envelope",
    "morphology"
]

with open("virus_blueprint_with_features.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(['Accession', 'Family', 'Virus_Name', 'Target_Protein',
                        'Danger_Score', 'Infects_Humans', 'Baltimore',
                        'Capsid', 'Envelope', 'Morphology'])

    for acc, info in enhanced_mapping.items():

        if isinstance(info, dict):
            row = [
                acc,
                info.get("family", ""),
                info.get("virus_name", ""),
                info.get("antigen", ""),
                info.get("risk_level", ""),
                info.get("baltimore", ""),
                info.get("capsid", ""),
                info.get("envelope", ""),
                info.get("morphology", "")
            ]

        else:
            print(f"Skipping malformed entry: {acc} -> {info}")
            continue

        writer.writerow(row)

print("CSV written successfully.")

Skipping malformed entry: NP_040896 -> ('Parvoviridae', 'Canine Parvovirus (CPV-2b Taiwan)', 'VP2', 2, 'Group II: ssDNA', 'Icosahedral', 'Naked', '20-26 nm, smallest known viruses')
CSV written successfully.


In [ ]:
import pandas as pd
import time

print("🛰️ Connecting to NCBI Servers...")

# We will create a fresh list to hold our populated rows
fully_downloaded_rows = []

# This loop goes through every single row in your blueprint table
for index, row in df_blueprint.iterrows():
    accession = row["Accession"]
    virus_name = row["Virus_Name"]

    print(f"📥 Fetching [{virus_name}] using Accession: {accession}...")

    # Call our NCBI downloader function
    sequence_string = fetch_protein_sequence(accession)

    if sequence_string:
        # Start with all columns from the current row (as a dictionary)
        row_dict = row.to_dict()

        # Add the downloaded sequence
        row_dict["Sequence"] = sequence_string

        fully_downloaded_rows.append(row_dict)
        print(f"   ✅ Successfully downloaded {len(sequence_string)} letters.")
    else:
        print(f"   ⚠️ Skipping {virus_name} due to download error.")

    # Pause for 1 second between downloads so NCBI doesn't get mad at us
    time.sleep(1.0)

# Convert our finished list into a final master DataFrame
df_loaded_sequences = pd.DataFrame(fully_downloaded_rows)

# Filter out any sequences that are too short (likely errors)
df_loaded_sequences = df_loaded_sequences[df_loaded_sequences["Sequence"].str.len() > 20]

# Force the target variable to be recognized strictly as clean integer categories
df_loaded_sequences['Danger_Score'] = df_loaded_sequences['Danger_Score'].astype(int)

# Ensure Infects_Humans is correctly mapped (in case some rows were missing)
df_loaded_sequences['Infects_Humans'] = df_loaded_sequences['Accession'].map(infects_humans_map).fillna(0).astype(int)

# Verify that you actually have all three classes represented in your data
print("📊 Current class balance in your dataset:")
print(df_loaded_sequences['Danger_Score'].value_counts())

# Export the complete dataset (all features + sequence)
df_loaded_sequences.to_csv("downloaded_virus_sequences.csv", index=False)

print(f"\n🎉 All Done! Prepared {len(df_loaded_sequences)} virus sequences for the AI model.")
print(f"Columns saved: {list(df_loaded_sequences.columns)}")

🛰️ Connecting to NCBI Servers...
📥 Fetching [Vaccinia] using Accession: YP_232997...
   ✅ Successfully downloaded 248 letters.
📥 Fetching [Mpox] using Accession: URK44321...
   ✅ Successfully downloaded 43 letters.
📥 Fetching [Smallpox] using Accession: NP_042078...
   ✅ Successfully downloaded 439 letters.
📥 Fetching [Human Parainfluenza 1] using Accession: NP_604441...
   ✅ Successfully downloaded 575 letters.
📥 Fetching [Measles (Edmonston)] using Accession: NP_004680...
   ✅ Successfully downloaded 715 letters.
📥 Fetching [Mumps (Jeryl Lynn)] using Accession: YP_009142751...
   ✅ Successfully downloaded 378 letters.
📥 Fetching [Nipah Virus] using Accession: NP_112026...
   ✅ Successfully downloaded 546 letters.
📥 Fetching [Hendra Virus] using Accession: NP_047111...
   ✅ Successfully downloaded 546 letters.
📥 Fetching [Coxsackievirus A9] using Accession: NP_741961...
   ✅ Successfully downloaded 64 letters.
📥 Fetching [Rhinovirus A] using Accession: NP_740523...
   ✅ Successfully d

In [ ]:
unfetched_viruses_accessions = df_loaded_sequences[~df_loaded_sequences['Accession'].isin(df_loaded_sequences['Accession'])]['Accession'].tolist()
print(f"Accessions from the blueprint that were not successfully fetched from NCBI: {len(unfetched_viruses_accessions)}")
print(unfetched_viruses_accessions)

Accessions from the blueprint that were not successfully fetched from NCBI: 0
[]


In [ ]:
import torch
from transformers import AutoTokenizer, EsmModel

print("🧠 Initializing Meta ESM-2 Smart Brain...")

# Step A: Load the specialized Tokenizer for ESM-2
# This maps letters like 'M' or 'A' to exact numeric tokens the brain understands
model_name = "facebook/esm2_t30_150M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step B: Load the actual pre-trained Neural Network brain
model = EsmModel.from_pretrained(model_name)
modeltest = EsmModel.from_pretrained(model_name)

# Step C: Check for GPU acceleration (makes things run 10x faster)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
modeltest = model.to(device)
model.eval() # Put the model in evaluation mode (turn off learning switches)
modeltest.eval()
print(f"✅ ESM-2 is fully loaded and running on your system's: [{str(device).upper()}]")


🧠 Initializing Meta ESM-2 Smart Brain...


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ ESM-2 is fully loaded and running on your system's: [CPU]


In [ ]:
import xgboost as xgb
print(xgb.__version__)

3.4.1


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE

# =============================================================================
# 1. Merge all metadata columns from df_blueprint
# =============================================================================
df_merged = df_loaded_sequences.copy()
print("Initial columns:", df_merged.columns.tolist())

# List all blueprint columns and add any that are missing from df_loaded_sequences
blueprint_columns = df_blueprint.columns.tolist()
missing_cols = [c for c in blueprint_columns if c not in df_merged.columns and c != "Sequence"]

if missing_cols:
    df_merged = df_merged.merge(
        df_blueprint[["Accession"] + missing_cols],
        on="Accession",
        how="left"
    )
    print(f"Merged missing columns: {missing_cols}")
else:
    print("No additional blueprint columns needed.")

print("Merged columns:", df_merged.columns.tolist())

# =============================================================================
# 2. Define numeric and categorical feature groups
# =============================================================================

# Numeric metadata columns to feed directly into the model
numeric_meta_cols = [
    "Infects_Humans",
    "CFR_Percent",
    "Incubation_Days",
    "Host_Range_Count",
    "Zoonotic_Potential",
    "cytoplasm",
    "nucleus",
]

# Binary one-hot columns that may already exist
binary_meta_cols = [
    c for c in df_merged.columns
    if c.startswith("cargo_") or c.startswith("persistence_")
]

available_numeric_meta_cols = [
    c for c in numeric_meta_cols + binary_meta_cols
    if c in df_merged.columns
]

# Categorical columns to one-hot encode
categorical_cols = [
    c for c in ["Family", "Baltimore", "Capsid", "Envelope", "Morphology", "Persistence_Type"]
    if c in df_merged.columns
]

# Avoid duplicating Persistence_Type if persistence_* one-hot columns already exist
if any(c.startswith("persistence_") for c in df_merged.columns):
    categorical_cols = [c for c in categorical_cols if c != "Persistence_Type"]

# Convert numeric metadata columns to float
for c in available_numeric_meta_cols:
    df_merged[c] = pd.to_numeric(df_merged[c], errors="coerce").fillna(0.0)

print("\nNumeric metadata columns:", available_numeric_meta_cols)
print("Categorical columns:", categorical_cols)

# =============================================================================
# 3. Helper: extra biochemical features
# =============================================================================
def extra_features(seq):
    seq = seq.upper()
    L = len(seq)
    aas = "ACDEFGHIKLMNPQRSTVWY"
    comp = [seq.count(a) / L for a in aas]
    pos = seq.count("R") + seq.count("K")
    neg = seq.count("D") + seq.count("E")
    charge = (pos - neg) / L

    analysed = ProteinAnalysis(seq)

    try:
        iep = analysed.isoelectric_point()
    except:
        iep = 7.0

    try:
        instability = analysed.instability_index()
    except:
        instability = 40.0

    try:
        aromaticity = analysed.aromaticity()
    except:
        aromaticity = 0.05

    try:
        gravy = analysed.gravy()
    except:
        gravy = 0.0

    try:
        helix, turn, sheet = analysed.secondary_structure_fraction()
    except:
        helix, turn, sheet = 0.3, 0.3, 0.4

    return [L] + comp + [charge, iep, instability, aromaticity, gravy, helix, turn, sheet]


# =============================================================================
# 4. Batch ESM-2 embedding + extra features + numeric metadata
# =============================================================================
sequences = df_merged["Sequence"].tolist()
labels = df_merged["Danger_Score"].tolist()
names = df_merged["Virus_Name"].tolist()

# Precompute numeric metadata matrix
if available_numeric_meta_cols:
    meta_matrix = df_merged[available_numeric_meta_cols].astype(float).to_numpy()
else:
    meta_matrix = np.zeros((len(df_merged), 0))

batch_size = 16
fingerprint_matrix = []
model.eval()

with torch.no_grad():
    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i:i + batch_size]

        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)

        mask = inputs["attention_mask"]
        hidden = outputs.last_hidden_state
        lengths = mask.sum(dim=1, keepdim=True).float()
        pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths
        embs = pooled.cpu().numpy()

        for j, seq in enumerate(batch_seqs):
            global_idx = i + j

            ef = np.array(extra_features(seq))
            meta = meta_matrix[global_idx]

            full = np.concatenate([embs[j], ef, meta])
            fingerprint_matrix.append(full)

# =============================================================================
# 5. Add one-hot encoded categorical features
# =============================================================================
X_numeric = np.array(fingerprint_matrix)

if categorical_cols:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    cat_encoded = encoder.fit_transform(df_merged[categorical_cols])
    X = np.hstack([X_numeric, cat_encoded])
else:
    X = X_numeric

y = np.array(labels)

print(f"\nFeature matrix shape (numeric only): {X_numeric.shape}")
print(f"Feature matrix shape (with categorical features): {X.shape}")

# =============================================================================
# 6. Stratified split and class weights
# =============================================================================
X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X, y, names, test_size=0.20, random_state=42, stratify=y
)

classes = np.unique(y_train)
class_weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

print("\nClass distribution (train):", np.bincount(y_train))
print("Class weights:", class_weight_dict)

# =============================================================================
# 7. XGBoost hyperparameter tuning
# =============================================================================
xgb = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    xgb,
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

grid.fit(X_train, y_train, sample_weight=sample_weights_train)

best_clf = grid.best_estimator_
print("\nBest parameters:", grid.best_params_)

cv_scores = cross_val_score(
    best_clf,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    params={"sample_weight": sample_weights_train}
)
print(f"Training CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ---------- Final test ----------
y_pred = best_clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"\n🎯 Test Accuracy: {test_acc*100:.1f}%")
print("\n📊 Classification Report (Test Set):")
print(classification_report(
    y_test,
    y_pred,
    target_names=["0 (Mild)", "1 (Medium)", "2 (High)"]
))

ModuleNotFoundError: No module named 'Bio'

In [ ]:
import xgboost as xgb
# Grab the very first virus sequence string from your freshly downloaded table
sample_sequence = df_loaded_sequences["Sequence"].iloc[0]
sample_name = df_loaded_sequences["Virus_Name"].iloc[0]

xgb_param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.3]
}
print(f"Testing translator on: {sample_name}")

# 1. Tokenize: Convert the text string into PyTorch tensors (math packets)
inputs = tokenizer(sample_sequence, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()} # Move data to the computing device

# 2. Translate: Run the letters through Meta's brain without calculating slopes (gradients)
with torch.no_grad():
    outputs = modeltest(**inputs)

# 3. Extract Fingerprint: Get the hidden states.
# We take the mathematical average (mean) across the whole sequence to get one steady fingerprint vector.
embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

print("\n✨ Translation complete! Look at what the brain produced:")
print(f"Shape of the fingerprint vector: {embeddings.shape}")
print(f"First 5 math numbers of the virus's new identity: {embeddings[:5]}")


In [ ]:
import numpy as np
import torch
from Bio.SeqUtils.ProtParam import ProteinAnalysis

print("🏭 Starting the AI Translation Assembly Line...")

# =============================================================================
# Helper: extra biochemical features (same as used in training)
# =============================================================================
def extra_features(seq):
    seq = seq.upper()
    L = len(seq)
    aas = "ACDEFGHIKLMNPQRSTVWY"
    comp = [seq.count(a) / L for a in aas]
    pos = seq.count("R") + seq.count("K")
    neg = seq.count("D") + seq.count("E")
    charge = (pos - neg) / L

    analysed = ProteinAnalysis(seq)

    try:
        iep = analysed.isoelectric_point()
    except:
        iep = 7.0

    try:
        instability = analysed.instability_index()
    except:
        instability = 40.0

    try:
        aromaticity = analysed.aromaticity()
    except:
        aromaticity = 0.05

    try:
        gravy = analysed.gravy()
    except:
        gravy = 0.0

    try:
        helix, turn, sheet = analysed.secondary_structure_fraction()
    except:
        helix, turn, sheet = 0.3, 0.3, 0.4

    return [L] + comp + [charge, iep, instability, aromaticity, gravy, helix, turn, sheet]

# =============================================================================
# Define numeric metadata columns to include
# =============================================================================
numeric_meta_cols = [
    "Infects_Humans",
    "CFR_Percent",
    "Incubation_Days",
    "Host_Range_Count",
    "Zoonotic_Potential",
    "cytoplasm",
    "nucleus",
]

# Add any one-hot columns that may exist
onehot_meta_cols = [
    col for col in df_loaded_sequences.columns
    if col.startswith("cargo_") or col.startswith("persistence_")
]

all_numeric_meta_cols = numeric_meta_cols + onehot_meta_cols

# Keep only columns actually present in df_loaded_sequences
available_numeric_meta_cols = [
    col for col in all_numeric_meta_cols
    if col in df_loaded_sequences.columns
]

# Convert them to float
for col in available_numeric_meta_cols:
    df_loaded_sequences[col] = pd.to_numeric(
        df_loaded_sequences[col], errors="coerce"
    ).fillna(0.0)

print(f"Metadata columns used: {available_numeric_meta_cols}")

# =============================================================================
# 1. Create lists to hold our numeric inputs (X) and danger answers (y)
# =============================================================================
fingerprint_matrix = []
danger_labels = []
tracking_names = []

# Turn off gradient calculations to save memory and make it run fast
with torch.no_grad():
    for index, row in df_loaded_sequences.iterrows():
        name = row["Virus_Name"]
        sequence = row["Sequence"]
        score = row["Danger_Score"]

        # A. Tokenization
        inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # B. Feed into ESM-2 model
        outputs = model(**inputs)

        # C. Mean-pool the last hidden state
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

        # D. Compute biochemical features
        biochem = np.array(extra_features(sequence))

        # E. Extract numeric metadata
        meta = df_loaded_sequences.loc[index, available_numeric_meta_cols].astype(float).to_numpy()

        # F. Concatenate all features
        full_vector = np.concatenate([embedding, biochem, meta])

        # G. Store
        fingerprint_matrix.append(full_vector)
        danger_labels.append(score)
        tracking_names.append(name)

        print(f"   🧬 Processed: {name:<30} -> Generated vector shape: {full_vector.shape}")

# =============================================================================
# 2. Convert lists into solid NumPy arrays
# =============================================================================
X = np.array(fingerprint_matrix)
y = np.array(danger_labels)

print("\n✨ Assembly complete! Your data shapes look like this:")
print(f"📊 Feature Matrix (X) shape: {X.shape}  <- ({X.shape[0]} viruses, each with {X.shape[1]} attributes)")
print(f"🎯 Target Labels (y) shape:   {y.shape}  <- ({len(y)} matching danger scores)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

# Ensure names is defined (same as used previously)
if 'names' not in dir():
    names = df_loaded_sequences['Virus_Name'].tolist()

# =============================================================================
# 1. Optional: one‑hot encode additional categorical columns if not already in X
# =============================================================================
# Family is always added by this script.
categorical_cols_to_add = ["Family"]

# You can also add other categorical columns that are still in df_loaded_sequences
# but have not yet been one‑hot encoded. For example:
# "Baltimore", "Capsid", "Envelope", "Morphology", "Persistence_Type"
for col in categorical_cols_to_add:
    if col not in df_loaded_sequences.columns:
        print(f"⚠️ Column '{col}' not found in df_loaded_sequences. Skipping.")
        continue

    # If column already exists as one‑hot, skip to avoid duplication
    if col == "Family" and any(c.startswith("fam_") for c in df_loaded_sequences.columns):
        continue  # already encoded, but we'll still create from original

    # Create dummies
    dummies = pd.get_dummies(df_loaded_sequences[col], prefix=col.lower())

    # Concatenate to X
    if 'X_augmented' not in dir():
        X_augmented = X.copy()
    X_augmented = np.hstack([X_augmented, dummies.values])
    print(f"Added {col} one‑hot features: {dummies.shape[1]} columns")

# If no categorical columns added, X_augmented is just X
if 'X_augmented' not in dir():
    X_augmented = X

print(f"Final feature matrix shape (with family): {X_augmented.shape}")

# =============================================================================
# 2. Stratified split
# =============================================================================
X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X_augmented, y, names, test_size=0.20, random_state=42, stratify=y
)

# =============================================================================
# 3. Class weights
# =============================================================================
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

# =============================================================================
# 4. Tune XGBoost (same grid as before)
# =============================================================================
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', tree_method='hist', n_jobs=-1)

param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(xgb, param_grid, cv=cv, scoring='accuracy', n_jobs=1, verbose=1)
grid.fit(X_train, y_train, sample_weight=sample_weights_train)

best_clf_direct = grid.best_estimator_
print("\nBest parameters (with family):", grid.best_params_)

# CV accuracy
cv_scores = cross_val_score(best_clf_direct, X_train, y_train, cv=cv,
                            scoring='accuracy',
                            params={'sample_weight': sample_weights_train})
print(f"Training CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Final test
y_pred = best_clf_direct.predict(X_test)
print(f"\n🎯 Test Accuracy (with family): {accuracy_score(y_test, y_pred)*100:.1f}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Mild", "Medium", "High"]))

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("✂️ Splitting data into a Training Group (80%) and a Final Exam Group (20%)...")

# Confirm feature dimensions
print(f"📐 Feature matrix shape: {X.shape}")
print(f"   (Embeddings + biochemical + metadata columns)")

# Split the data cleanly so the model doesn't get to cheat by seeing test questions early
X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X, y, tracking_names, test_size=0.20, random_state=42, stratify=y
)

# Optional: compute class weights to handle imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

print("🌲 Initializing and Training the Custom XGBoost Brain Modification...")

# Create the custom head model
classifier = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist',      # faster
    n_jobs=-1                # use all CPU cores
)

# Train with sample weights (optional, but helps if classes are imbalanced)
classifier.fit(X_train, y_train, sample_weight=sample_weights_train)
print("   🏆 Training finished successfully!")

print("\n📝 Giving the Model its Final Exam...")
# Ask the trained model to guess the danger scores for the test group it has never seen before
y_predictions = classifier.predict(X_test)

# Calculate how well it did
final_accuracy = accuracy_score(y_test, y_predictions)
print(f"🎯 Final Accuracy Score: {final_accuracy * 100:.1f}%\n")

# Print out a breakdown of performance for each danger tier (0, 1, and 2)
print("📊 Comprehensive Performance Report:")
print(classification_report(y_test, y_predictions, target_names=["0 (Mild)", "1 (Medium)", "2 (High)"]))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ============================================================
# 0. Use your current X, y (adjust if different)
# ============================================================
# X should be the final feature matrix you want to use
# y should be the Danger_Score labels
# If you have a different variable name, change it here.
X_data = X   # replace with X_with_fam_direct or X_stacked if needed
y_data = y

# ============================================================
# 1. Train/test split (same random state for reproducibility)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.20, random_state=42, stratify=y_data
)

# Class weights for training
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

# ============================================================
# 2. Define the three models with their best hyperparameters
# ============================================================
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1,
    colsample_bytree=1.0,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=3,
    n_estimators=200,
    subsample=1.0
)

rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=50,          # from your earlier tuning
    max_depth=3,              # from your earlier tuning
    criterion='gini'
)

lgb_model = LGBMClassifier(
    random_state=42,
    verbose=-1,
    learning_rate=0.01,       # from your earlier tuning
    n_estimators=100,         # from your earlier tuning
    num_leaves=31,            # from your earlier tuning
    min_child_samples=5       # from your earlier tuning
)

# ============================================================
# 3. Build voting ensemble (soft voting)
# ============================================================
ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('rf', rf_model), ('lgb', lgb_model)],
    voting='soft',   # average predicted probabilities
    weights=[2, 1, 1]   # give XGBoost more weight (optional)
)

# ============================================================
# 4. Train ensemble with sample weights
# ============================================================
# VotingClassifier doesn't support sample_weight directly for all models.
# We'll train each base model separately with sample_weight, then combine manually.
xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
rf_model.fit(X_train, y_train, sample_weight=sample_weights_train)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)

# Manually compute soft-voting probabilities on test set
probs_xgb = xgb_model.predict_proba(X_test)
probs_rf = rf_model.predict_proba(X_test)
probs_lgb = lgb_model.predict_proba(X_test)

# Weighted average (weights must match order)
weights = np.array([2, 1, 1], dtype=float)
weights = weights / weights.sum()
ensemble_probs = (weights[0] * probs_xgb +
                  weights[1] * probs_rf +
                  weights[2] * probs_lgb)

y_pred_ensemble = np.argmax(ensemble_probs, axis=1)

# ============================================================
# 5. Evaluate ensemble
# ============================================================
test_acc = accuracy_score(y_test, y_pred_ensemble)
print(f"\n🎯 Ensemble Test Accuracy: {test_acc*100:.1f}%")
print("\n📊 Ensemble Classification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=["Mild","Medium","High"]))

# Macro AUC
auc = roc_auc_score(y_test, ensemble_probs, multi_class='ovr', average='macro')
print(f"Ensemble Macro AUC: {auc:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report, accuracy_score

# Use the ensemble_probs from the previous cell (or any probability matrix)
probs = ensemble_probs   # shape (n_test, 3)

# Get the original class predictions (argmax)
y_pred_default = np.argmax(probs, axis=1)

# We'll tune the threshold for High class (index 2)
thresholds = np.arange(0.15, 0.5, 0.01)
best_thresh = 0.33
best_score = 0

for t in thresholds:
    y_pred_tuned = y_pred_default.copy()
    high_mask = probs[:, 2] > t
    y_pred_tuned[high_mask] = 2
    # Use macro F1 as optimisation metric
    score = f1_score(y_test, y_pred_tuned, average='macro')
    if score > best_score:
        best_score = score
        best_thresh = t

print(f"Optimised threshold for High: {best_thresh:.2f} (macro F1 = {best_score:.3f})")

# Final prediction with tuned threshold
y_pred_final = y_pred_default.copy()
y_pred_final[probs[:, 2] > best_thresh] = 2

print("\n🎯 Tuned Ensemble Accuracy:", accuracy_score(y_test, y_pred_final)*100, "%")
print("\n📊 Tuned Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=["Mild","Medium","High"]))

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# ============================================================
# Use the XGBoost model from the ensemble (xgb_model)
# ============================================================
best_model_for_shap = xgb_model

# Ensure X_test shape matches model
if X_test.shape[1] != best_model_for_shap.n_features_in_:
    print(f"⚠️ Dimension mismatch: model expects {best_model_for_shap.n_features_in_}, X_test has {X_test.shape[1]}")
    X_test = X_test[:, :best_model_for_shap.n_features_in_]

# ============================================================
# SHAP explanation
# ============================================================
explainer = shap.TreeExplainer(best_model_for_shap)
shap_values = explainer.shap_values(X_test)   # (n_samples, n_features, n_classes)

class_labels = {0: "Mild", 1: "Medium", 2: "High"}
output_dir = "shap_ensemble_final"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# Loop over classes and create aggregated bar charts
# ============================================================
for class_idx, class_name in class_labels.items():
    if len(shap_values.shape) == 3:
        class_shap = shap_values[:, :, class_idx]
    else:
        class_shap = shap_values[class_idx]

    # Per‑virus top‑15 features
    all_top_features = []
    for i in range(len(X_test)):
        sample_shap = class_shap[i]
        abs_shap = np.abs(sample_shap)
        top15_idx = np.argsort(abs_shap)[-15:][::-1]
        top15_vals = abs_shap[top15_idx]
        all_top_features.append(pd.DataFrame({
            'Feature_Index': top15_idx,
            'SHAP_Absolute': top15_vals,
            'Virus': [names_test[i]]*15
        }))

    global_features = pd.concat(all_top_features, ignore_index=True)

    # Aggregated top‑15 across all samples
    mean_importance = (global_features.groupby('Feature_Index')['SHAP_Absolute']
                       .mean().sort_values(ascending=False).head(15))

    # Publication bar chart
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(mean_importance)))
    plt.barh([f"F{int(idx)}" for idx in mean_importance.index[::-1]],
             mean_importance.values[::-1], color=colors)
    plt.title(f"Most Influential Features for '{class_name}' Severity")
    plt.xlabel("Mean Absolute SHAP Value")
    plt.ylabel("Feature Index")
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"shap_{class_name}.png"), dpi=300)
    plt.show()

print(f"✅ SHAP analysis complete. Charts saved to '{output_dir}/'")

In [ ]:
def predict_new_virus_severity(raw_sequence, virus_label="Unknown Variant"):
    """
    Passes a raw text protein string through the loaded tokenizer, ESM-2 model,
    and trained XGBoost classifier to return a real-time risk assessment.
    """
    # 1. Standardize and format string input
    clean_seq = raw_sequence.strip().upper().replace(" ", "")

    # 2. Extract mathematical vector embedding via ESM-2
    inputs = tokenizer(clean_seq, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    vector = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().reshape(1, -1)

    # 3. Compute predicted class and raw confidence probabilities
    predicted_class = classifier.predict(vector)[0]
    probabilities = classifier.predict_proba(vector)[0]

    tier_names = {0: "0 (Mild Risk)", 1: "1 (Moderate Risk)", 2: "2 (High Clinical Severity/Mortality)"}

    print(f"🔮 Real-Time Risk Analysis for: {virus_label}")
    print(f"   🧬 Input Sequence Length: {len(clean_seq)} amino acids")
    print(f"   🏆 Assigned Designation: Tier {tier_names[predicted_class]}")
    print(f"   📊 Confidence Metrics: [Tier 0: {probabilities[0]:.2%}] | [Tier 1: {probabilities[1]:.2%}] | [Tier 2: {probabilities[2]:.2%}]\n")

    return predicted_class, probabilities

# Quick Execution Test using an arbitrary mock sequence snippet
test = predict_new_virus_severity("MTSVVVVAVALLLAAAGRAA", "Hypothetical Mutant Strain")


In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def extra_features(seq):
    seq = seq.upper()
    L = len(seq)
    aas = "ACDEFGHIKLMNPQRSTVWY"
    comp = [seq.count(a) / L for a in aas]
    pos = seq.count("R") + seq.count("K")
    neg = seq.count("D") + seq.count("E")
    charge = (pos - neg) / L

    analysed = ProteinAnalysis(seq)

    try:
        iep = analysed.isoelectric_point()
    except:
        iep = 7.0

    try:
        instability = analysed.instability_index()
    except:
        instability = 40.0

    try:
        aromaticity = analysed.aromaticity()
    except:
        aromaticity = 0.05

    try:
        gravy = analysed.gravy()
    except:
        gravy = 0.0

    try:
        helix, turn, sheet = analysed.secondary_structure_fraction()
    except:
        helix, turn, sheet = 0.3, 0.3, 0.4

    return [L] + comp + [charge, iep, instability, aromaticity, gravy, helix, turn, sheet]


def predict_new_virus_severity(
    raw_sequence,
    virus_label="Unknown Variant",
    infect_humans=1,
    cfr_percent=5.0,
    incubation_days=7.0,
    host_range_count=3,
    zoonotic=1,
    cytoplasm=1,
    nucleus=0,
    cargo_env=0,
    cargo_capsid=1,
    cargo_other=0,
    pers_acute=1,
    pers_chronic=0,
    pers_latent=0
):
    """
    Predict danger tier for a new protein sequence.

    The classifier expects 682 features:
        [ESM‑2 embedding (640), biochemical (29), metadata (7), binary meta (6)]

    Optional metadata parameters are provided with reasonable defaults.
    If you know the actual values, pass them to improve accuracy.
    """
    # 1. Standardize and format string input
    clean_seq = raw_sequence.strip().upper().replace(" ", "")

    # 2. ESM‑2 embedding
    inputs = tokenizer(clean_seq, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

    # 3. Biochemical features (29 dimensions)
    biochem = np.array(extra_features(clean_seq))

    # 4. Metadata vector (13 dimensions – must match training order)
    meta = np.array([
        infect_humans,
        cfr_percent,
        incubation_days,
        host_range_count,
        zoonotic,
        cytoplasm,
        nucleus,
        cargo_env,
        cargo_capsid,
        cargo_other,
        pers_acute,
        pers_chronic,
        pers_latent
    ], dtype=float)

    # 5. Concatenate in the exact order used during training
    full_vector = np.concatenate([embedding, biochem, meta]).reshape(1, -1)

    # 6. Sanity check
    expected_features = classifier.n_features_in_
    if full_vector.shape[1] != expected_features:
        raise ValueError(
            f"Feature mismatch: got {full_vector.shape[1]}, expected {expected_features}. "
            "Check metadata dimension or model training features."
        )

    # 7. Predict
    predicted_class = classifier.predict(full_vector)[0]
    probabilities = classifier.predict_proba(full_vector)[0]

    tier_names = {
        0: "0 (Mild Risk)",
        1: "1 (Moderate Risk)",
        2: "2 (High Clinical Severity/Mortality)"
    }

    print(f"🔮 Real-Time Risk Analysis for: {virus_label}")
    print(f"   🧬 Input Sequence Length: {len(clean_seq)} amino acids")
    print(f"   🏆 Assigned Designation: Tier {tier_names[predicted_class]}")
    print(f"   📊 Confidence Metrics: [Tier 0: {probabilities[0]:.2%}] | "
          f"[Tier 1: {probabilities[1]:.2%}] | [Tier 2: {probabilities[2]:.2%}]\n")

    return predicted_class, probabilities


# Quick test
test = predict_new_virus_severity("MTSVVVVAVALLLAAAGRAA", "Hypothetical Mutant Strain")

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# ========== 1. SHAP explanation (class 2 = High Severity) ==========
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test)          # shape: (n_samples, 320, n_classes)

if len(shap_values.shape) == 3:
    tier2_shap = shap_values[:, :, 2]                # (n_samples, 320)
else:
    tier2_shap = shap_values[2]

# ========== 2. Create output directory for your paper’s assets ==========
output_dir = "shap_analysis"
os.makedirs(output_dir, exist_ok=True)

# ========== 3. Loop over every test sample and extract top-15 features ==========
all_top_features = []   # will hold per-sample DataFrames
per_sample_summary = [] # for a quick look-up table

for i in range(len(X_test)):
    sample_shap = tier2_shap[i]
    abs_shap = np.abs(sample_shap)

    # Indices of the 15 largest absolute SHAP values
    top15_idx = np.argsort(abs_shap)[-15:][::-1]
    top15_vals = abs_shap[top15_idx]

    # Build a dataframe for this sample
    df_sample = pd.DataFrame({
        'Feature_Index': top15_idx,
        'SHAP_Absolute': top15_vals
    })
    df_sample['Virus'] = names_test[i]     # names_test from your train_test_split
    all_top_features.append(df_sample)

    # Also keep a compact summary (top 5 only, for a table in the paper)
    top5_str = ", ".join([f"Dim{idx}({val:.3f})" for idx, val in zip(top15_idx[:5], top15_vals[:5])])
    per_sample_summary.append({'Virus': names_test[i], 'Top5_Dims': top5_str})

# Combine all samples into one big table
global_features = pd.concat(all_top_features, ignore_index=True)

# ========== 4. Save per‑sample top features to a CSV (very useful for the paper) ==========
global_features.to_csv(os.path.join(output_dir, "per_virus_top15_shap_features.csv"), index=False)

# Also save the quick summary table
summary_df = pd.DataFrame(per_sample_summary)
summary_df.to_csv(os.path.join(output_dir, "per_virus_top5_summary.csv"), index=False)

# ========== 5. Aggregate: mean importance of each embedding dimension across all viruses ==========
mean_importance = (
    global_features.groupby('Feature_Index')['SHAP_Absolute']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

# Save the aggregated results as a table too
mean_importance_df = mean_importance.reset_index()
mean_importance_df.columns = ['Embedding_Dimension', 'Mean_Absolute_SHAP']
mean_importance_df.to_csv(os.path.join(output_dir, "aggregated_top15_dimensions.csv"), index=False)

# ========== 6. Publication‑ready plot ==========
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(mean_importance)))
plt.barh(
    [f"Dimension {int(idx)}" for idx in mean_importance.index[::-1]],
    mean_importance.values[::-1],
    color=colors
)
plt.title("Most Influential ESM‑2 Embedding Dimensions for\n'High Severity' Predictions", fontsize=12, fontweight='bold')
plt.xlabel("Mean Absolute SHAP Value (across all test viruses)", fontsize=10)
plt.ylabel("Embedding Dimension Index", fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "shap_top_dimensions.png"), dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ All SHAP analysis files saved to '{output_dir}/'")
print("   - per_virus_top15_shap_features.csv")
print("   - per_virus_top5_summary.csv")
print("   - aggregated_top15_dimensions.csv")
print("   - shap_top_dimensions.png")

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# ========== 1. SHAP explanation ==========
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test)   # (n_samples, 320, n_classes)

class_labels = {0: "Mild", 1: "Medium", 2: "High"}

# ========== 2. Create output directory tree ==========
output_dir = "shap_analysis_full"
os.makedirs(output_dir, exist_ok=True)

chart_dirs = {}
for cls in class_labels.values():
    path = os.path.join(output_dir, f"per_virus_charts_{cls}")
    os.makedirs(path, exist_ok=True)
    chart_dirs[cls] = path

# ========== 3. Main loop over severity classes ==========
for class_idx, class_name in class_labels.items():
    # Extract SHAP values for this class
    if len(shap_values.shape) == 3:
        class_shap = shap_values[:, :, class_idx]
    else:
        class_shap = shap_values[class_idx]

    # ---- 3a. Collect per‑sample top features & create individual charts ----
    all_top_features = []
    per_sample_summary = []

    for i in range(len(X_test)):
        sample_shap = class_shap[i]
        abs_shap = np.abs(sample_shap)

        top15_idx = np.argsort(abs_shap)[-15:][::-1]
        top15_vals = abs_shap[top15_idx]

        df_sample = pd.DataFrame({
            'Feature_Index': top15_idx,
            'SHAP_Absolute': top15_vals
        })
        df_sample['Virus'] = names_test[i]
        all_top_features.append(df_sample)

        top5_str = ", ".join([f"Dim{idx}({val:.3f})" for idx, val in zip(top15_idx[:5], top15_vals[:5])])
        per_sample_summary.append({'Virus': names_test[i], 'Top5_Dims': top5_str})

        # Save per‑virus chart (not displayed to keep notebook output clean)
        virus_name_clean = names_test[i].replace(" ", "_").replace("/", "_")
        plt.figure(figsize=(8, 5))
        colors = plt.cm.viridis(np.linspace(0.2, 0.8, 15))
        plt.barh(
            [f"Dim {int(idx)}" for idx in top15_idx[::-1]],
            top15_vals[::-1],
            color=colors
        )
        plt.title(f"Top ESM‑2 Dimensions for '{class_name}' Severity\n{names_test[i]}", fontsize=11, fontweight='bold')
        plt.xlabel("Absolute SHAP Value", fontsize=9)
        plt.ylabel("Embedding Dimension Index", fontsize=9)
        plt.grid(axis='x', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(chart_dirs[class_name], f"{virus_name_clean}.png"), dpi=200)
        plt.close()   # Do NOT display per‑virus charts; remove this line to see them all

    global_features = pd.concat(all_top_features, ignore_index=True)

    # ---- 3b. Save per‑sample CSVs ----
    global_features.to_csv(os.path.join(output_dir, f"per_virus_top15_shap_{class_name}.csv"), index=False)

    # ---- 3c. Per‑sample summary LaTeX ----
    summary_df = pd.DataFrame(per_sample_summary)
    summary_df.to_csv(os.path.join(output_dir, f"per_virus_top5_summary_{class_name}.csv"), index=False)

    latex_table = summary_df.to_latex(index=False, caption=f"Top 5 most influential ESM‑2 dimensions for each virus ({class_name} severity).", label=f"tab:top5_{class_name}")
    with open(os.path.join(output_dir, f"per_virus_top5_{class_name}.tex"), "w") as f:
        f.write(latex_table)

    # ---- 3d. Aggregated top‑15 dimensions ----
    mean_importance = (
        global_features.groupby('Feature_Index')['SHAP_Absolute']
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )

    mean_importance_df = mean_importance.reset_index()
    mean_importance_df.columns = ['Embedding_Dimension', 'Mean_Absolute_SHAP']
    mean_importance_df.to_csv(os.path.join(output_dir, f"aggregated_top15_dimensions_{class_name}.csv"), index=False)

    latex_agg = mean_importance_df.to_latex(index=False, float_format="%.4f", caption=f"Globally most important ESM‑2 embedding dimensions for {class_name} severity.", label=f"tab:agg_{class_name}")
    with open(os.path.join(output_dir, f"aggregated_top15_{class_name}.tex"), "w") as f:
        f.write(latex_agg)

    # ---- 3e. Aggregated bar chart (DISPLAY IT) ----
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(mean_importance)))
    plt.barh(
        [f"Dim {int(idx)}" for idx in mean_importance.index[::-1]],
        mean_importance.values[::-1],
        color=colors
    )
    plt.title(f"Most Influential ESM‑2 Dimensions for '{class_name}' Severity", fontsize=12, fontweight='bold')
    plt.xlabel("Mean Absolute SHAP Value (across all test viruses)", fontsize=10)
    plt.ylabel("Embedding Dimension Index", fontsize=10)
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"shap_top_dimensions_{class_name}.png"), dpi=300)
    plt.show()   # <--- This prints the chart directly in your notebook

print(f"✅ Full SHAP analysis exported to '{output_dir}/'")
print("📁 Contains:")
print("   ├── per_virus_charts_Mild/   (individual PNG for each test virus)")
print("   ├── per_virus_charts_Medium/")
print("   ├── per_virus_charts_High/")
print("   ├── per_virus_top15_shap_*.csv")
print("   ├── per_virus_top5_*.tex       (ready-to-use LaTeX tables)")
print("   ├── aggregated_top15_*.tex     (LaTeX for aggregated dimensions)")
print("   └── shap_top_dimensions_*.png  (aggregated bar charts – also shown above)")

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import json

print("⚙️ Initiating Hyperparameter Optimization Loops...")

# Ensure cross-validation is stratified to handle small dataset splits evenly
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- Tuning Loop A: Random Forest Framework ---
print("\n🌲 Tuning Random Forest (Testing Estimators & Split Criteria)...")
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
    'criterion': ['gini', 'entropy']
}

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1
)
rf_grid_search.fit(X_train, y_train)
best_rf_model = rf_grid_search.best_estimator_
print(f"   🏆 Best RF Parameters: {rf_grid_search.best_params_}")

# --- Tuning Loop B: LightGBM Framework ---
print("\n⚡ Tuning LightGBM Framework (Testing Learning Rates & Regularization)...")
lgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [2, 5, 10]
}

lgb_grid_search = GridSearchCV(
    estimator=LGBMClassifier(random_state=42, verbose=-1),
    param_grid=lgb_param_grid,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1
)
lgb_grid_search.fit(X_train, y_train)
best_lgb_model = lgb_grid_search.best_estimator_
print(f"   🏆 Best LightGBM Parameters: {lgb_grid_search.best_params_}")